# NHANES 1999-2018 - Data Merging & Harmonisation Pipeline

**Purpose.** Build a single, analysis-ready dataset of U.S. participants from ten consecutive
NHANES cycles (1999-2000 through 2017-2018). Each cycle is processed independently into a
harmonised per-cycle CSV, and the ten CSVs are then stacked into one combined file used for
downstream cardiometabolic phenotyping and survival analysis.

**Inputs.** Per cycle, a set of NHANES component files in SAS transport (`.XPT`) format:
demographics (DEMO), body measures (BMX), blood pressure (BPX), lipids (HDL / TRIGLY, or the
older LAB13 files), glycohaemoglobin (GHB / LAB10), plasma glucose & insulin (GLU / LAB10AM /
INS), standard biochemistry (BIOPRO / LAB18 / L40), the fasting questionnaire (FASTQX / PH),
the oral glucose tolerance test (OGTT), and the diabetes (DIQ), smoking (SMQ), alcohol (ALQ)
and physical-activity (PAQ) questionnaires. Files are matched to a cycle by their letter
suffix (none, `_B`, `_C`, ... `_J`).

**Outputs.** One clean CSV per cycle (`nhanes_YYYY_YYYY_clean.csv`) and one combined CSV
(`nhanes_1999_2018_combined_new.csv`, 101,316 rows x 32 columns). Every row is keyed by the
NHANES participant id `SEQN`.

**How to run.** Update the `BASE_DIR` path in each cycle cell (and the read paths in the
combine cell) to point at your local NHANES folders, make sure the output folder
`nhanes_data_csvm/` exists, then run the cells top to bottom. Requires `pandas` and `numpy`;
`.XPT` files are read with `pandas.read_sas`.

---

## Why the pipeline is shaped this way

Each cycle is handled in its own self-contained cell rather than by a single loop. That is
deliberate: NHANES renames variables and moves them between component files from cycle to
cycle, so the *selection* step genuinely differs each time even though the *derivation* logic
is the same. Keeping one cell per cycle makes each cycle independently runnable and makes the
cycle-specific quirks visible at the point where they matter, at the cost of some repetition.

The merge strategy is the same everywhere: start from `demo` (one row per participant) and
`left`-join every other component on `SEQN`. A left join is the right choice because most
components cover only a subsample (only fasting participants have insulin, only a subset had
an OGTT), and a left join preserves every demographic record while leaving unmeasured values
as `NaN` rather than silently dropping participants.

---

## Derived variables & methods (applied identically in every cycle unless noted)

- **Blood pressure.** `Systolic_BP` / `Diastolic_BP` are the row-wise means of up to four
  readings (`BPXSY1-4`, `BPXDI1-4`), ignoring missing values. Averaging repeated readings
  reduces measurement noise and white-coat effects; `skipna=True` means a participant with
  only two valid readings still gets a mean rather than `NaN`.
- **eGFR.** CKD-EPI 2021 creatinine equation (the race-free revision).
- **HOMA-IR / HOMA-B.** Computed only for participants fasting >= 8 hours, because both
  indices are defined on fasting glucose and insulin and are meaningless post-prandially.
  Glucose is auto-converted mg/dL -> mmol/L when its median exceeds 30 (a units guard: NHANES
  ships both `LBXGLU` in mg/dL and `LBDGLUSI` in mmol/L, and a median above 30 can only mean
  mg/dL). HOMA-IR = (glucose x insulin) / 22.5; HOMA-B = (20 x insulin) / (glucose - 3.5),
  restricted to glucose > 3.5 to avoid a zero or negative denominator.
- **Diabetes duration.** current age - age at diagnosis, floored at 0, only for confirmed
  diabetics (`DIQ010 == 1`) with a plausible diagnosis age (< 120). The `< 120` filter removes
  NHANES sentinel codes (777 = Refused, 999 = Don't know) that would otherwise be treated as
  real ages; the `clip(lower=0)` guards against recall inconsistencies where the reported
  diagnosis age exceeds current age.
- **Smoking / Alcohol / Physical activity.** Recoded to Ideal / Intermediate / Poor (or
  consumer / non-consumer) categories, broadly following the AHA cardiovascular-health scoring
  idea. *The exact source variables and the physical-activity rule change across cycles - see
  Cross-cycle harmonisation below.*
- **Ethnicity.** `RIDRETH1` integer codes mapped to text labels for readability.

---

## Cross-cycle harmonisation (why the cells are not identical)

| Concept | Variable / rule by cycle |
| --- | --- |
| Creatinine | `LBXSCR` in most cycles; `LBDSCR` in 2001-2002 |
| HDL | `LBDHDL` (1999-2002), `LBXHDD` (2003-2004), `LBDHDD` (2005-2018) |
| Fasting glucose | `LBXGLUSI` (1999-2002), `LBDGLUSI` (2003-2018) |
| Insulin & fasting hours file | in the glucose/fasting files through 2011-2012; in `INS_H`/`INS_I` for 2013-2016; in 2017-2018 fasting hours are in `FASTQX_J` and insulin in `INS_J` |
| Alcohol | `ALQ100` (1999-2000), `ALD100` (2001-2002), `ALQ101` (2003-2016), derived from `ALQ121` (2017-2018, after `ALQ101` was dropped) |
| Age at diagnosis | `DIQ040Q` (1999-2000), `DID040Q` (2001-2004), `DID040` (2005-2018) |
| Physical activity rule | binary activity items (`PAD200/320/440/460`) for 1999-2006 vs MVPA-minutes >= 150 with sentinel-code cleaning (`PAQ610...`/`PAD615...`) for 2007-2018 |
| OGTT (`Two_hour_glucose`) | merged only for 2005-2016; absent for 1999-2004 and not carried into the 2017-2018 final table |

Because the two physical-activity rules are **not equivalent**, `Physical_activity` is not
defined on the same basis across the whole 1999-2018 span; the same caveat applies to the
2017-2018 alcohol derivation. Anyone using these two variables in a pooled analysis should
either restrict to a consistent span or adjust for cycle.

---

## Pipeline structure

Cells 1-10 each build one cycle's clean CSV (identical structure, cycle-specific variable
names). Cell 11 stacks the ten CSVs into the combined dataset. Cell 12 prints the combined
schema.

> Cell outputs have been cleared so the notebook is light and diffs cleanly. Re-run top to
> bottom to regenerate them.


## Cycle 1: NHANES 1999-2000

- Component files: **no suffix** (`DEMO.XPT`, `BMX.XPT`, ...). Lipids and glucose come from the older `LAB13`/`LAB10` families.
- eGFR: CKD-EPI **2021**. Physical activity: **binary-item rule** (`PAD200/320/440/460`), where Ideal requires both aerobic and strength activity.
- Creatinine `LBXSCR`; fasting glucose `LBXGLUSI`; HDL `LBDHDL`; alcohol `ALQ100`.
- REVIEW: age at diagnosis is taken from `DIQ040Q` - confirm this is the numeric age rather than a qualifier (see Known issues #2).

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set the directory containing all .XPT files
BASE_DIR = Path(r"C:\Users\...\...\...\...\1999-2000")

# Load required files
demo     = pd.read_sas(BASE_DIR / "DEMO.XPT")
bmx      = pd.read_sas(BASE_DIR / "BMX.XPT")
bpx      = pd.read_sas(BASE_DIR / "BPX.XPT")
hdl      = pd.read_sas(BASE_DIR / "LAB13.XPT")
trig     = pd.read_sas(BASE_DIR / "LAB13AM.XPT")
hba1c    = pd.read_sas(BASE_DIR / "LAB10.XPT")
glu      = pd.read_sas(BASE_DIR / "LAB10AM.XPT")
creat    = pd.read_sas(BASE_DIR / "LAB18.XPT")
fast     = pd.read_sas(BASE_DIR / "PH.XPT")
diabetes = pd.read_sas(BASE_DIR / "DIQ.XPT")
smoking  = pd.read_sas(BASE_DIR / "SMQ.XPT")
alcohol  = pd.read_sas(BASE_DIR / "ALQ.XPT")
activity = pd.read_sas(BASE_DIR / "PAQ.XPT")

# ── Select Relevant Features ──────────────────────────────────────────────────
demo     = demo[["SEQN", "RIDAGEYR", "RIAGENDR", "RIDEXPRG", "RIDRETH1", "DMDEDUC2", "INDFMPIR"]]
bmx      = bmx[["SEQN", "BMXBMI"]]
bpx      = bpx[["SEQN", "BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4",
                        "BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]]
hba1c    = hba1c[["SEQN", "LBXGH"]]
trig     = trig[["SEQN", "LBDLDL", "LBXTR"]]
hdl      = hdl[["SEQN", "LBDHDL"]]
glu      = glu[["SEQN", "LBXGLUSI", "LBXIN"]]
creat    = creat[["SEQN", "LBXSCR", "LBDSGLSI"]]
fast     = fast[["SEQN", "PHAFSTHR"]]

diabetes = diabetes[["SEQN", "DIQ010", "DIQ040Q", "DIQ050", "DIQ070"]]

smoking  = smoking[["SEQN", "SMQ020", "SMQ040"]]
alcohol  = alcohol[["SEQN", "ALQ100"]]
activity = activity[["SEQN", "PAD200", "PAD320", "PAD440", "PAD460"]]

# ── Mean Blood Pressure ───────────────────────────────────────────────────────
bpx["SBP"] = bpx[["BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4"]].mean(axis=1, skipna=True)
bpx["DBP"] = bpx[["BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]].mean(axis=1, skipna=True)

# ── eGFR CKD-EPI (2021) ───────────────────────────────────────────────────────
def compute_egfr_2021(creatinine_mgdl, age, gender):
    age             = pd.to_numeric(age, errors='coerce')
    gender          = pd.to_numeric(gender, errors='coerce')
    creatinine_mgdl = pd.to_numeric(creatinine_mgdl, errors='coerce')

    k          = np.where(gender == 1, 0.9, 0.7)
    alpha      = np.where(gender == 1, -0.302, -0.241)
    sex_coeff  = np.where(gender == 1, 1.0, 1.012)

    min_scr = np.minimum(creatinine_mgdl / k, 1)
    max_scr = np.maximum(creatinine_mgdl / k, 1)

    egfr = 142 * (min_scr ** alpha) * (max_scr ** -1.200) * (0.9938 ** age) * sex_coeff
    return egfr

# ── HOMA-IR and HOMA-B ────────────────────────────────────────────────────────
def compute_homa(glu, ins, fasting_hours):
    glu           = pd.to_numeric(glu, errors='coerce')
    ins           = pd.to_numeric(ins, errors='coerce')
    # BUG FIX 2: The original mask used the raw `fasting_hours` Series before
    #            numeric conversion, risking silent failures on non-numeric input.
    #            Use the converted `fast` Series for the comparison.
    fast          = pd.to_numeric(fasting_hours, errors='coerce')

    if glu.median(skipna=True) > 30:   # Likely mg/dL → convert to mmol/L
        glu = glu / 18.018

    mask_fasting  = fast >= 8          # ← fixed: was `fasting_hours >= 8`
    homa_ir = np.where(mask_fasting, (glu * ins) / 22.5, np.nan)
    homa_b  = np.where(mask_fasting & (glu > 3.5), (20 * ins) / (glu - 3.5), np.nan)
    return homa_ir, homa_b

# ── Merge all DataFrames on SEQN ──────────────────────────────────────────────
df = (
    demo
    .merge(bmx,                       on="SEQN", how="left")
    .merge(bpx[["SEQN", "SBP", "DBP"]], on="SEQN", how="left")
    .merge(hba1c,                     on="SEQN", how="left")
    .merge(trig,                      on="SEQN", how="left")
    .merge(glu,                       on="SEQN", how="left")
    .merge(hdl,                       on="SEQN", how="left")
    .merge(creat,                     on="SEQN", how="left")
    .merge(fast,                      on="SEQN", how="left")
    .merge(diabetes,                  on="SEQN", how="left")
    .merge(smoking,                   on="SEQN", how="left")
    .merge(alcohol,                   on="SEQN", how="left")
    .merge(activity,                  on="SEQN", how="left")
)

# ── Derived Variables ─────────────────────────────────────────────────────────
df["eGFR"]              = compute_egfr_2021(df["LBXSCR"], df["RIDAGEYR"], df["RIAGENDR"])
df["HOMA_IR"], df["HOMA_B"] = compute_homa(df["LBXGLUSI"], df["LBXIN"], df["PHAFSTHR"])

# ── Diabetes Duration Workflow ────────────────────────────────────────────────
# Step 1 – Identify participants with confirmed diabetes (DIQ010 = 1)
has_diabetes = df["DIQ010"] == 1

# Step 2 – Extract DIQ040G (age at diagnosis); coerce any coded non-responses
#           (e.g. 999 = "Don't know", 777 = "Refused") to NaN before computing
age_at_dx = pd.to_numeric(df["DIQ040Q"], errors='coerce')
age_at_dx = age_at_dx.where(age_at_dx < 120)   # removes implausible sentinel values

# Step 3 – Subtract age at diagnosis from current age
current_age = pd.to_numeric(df["RIDAGEYR"], errors='coerce')

# Step 4 – Store as DIAB_DUR; only populated for confirmed diabetic participants
df["DIAB_DUR"] = np.where(
    has_diabetes & age_at_dx.notna(),
    (current_age - age_at_dx).clip(lower=0),   # floor at 0 to prevent negatives
    np.nan
)

# ── Smoking Category ──────────────────────────────────────────────────────────
df["Smoking_category"] = np.select(
    [
        df["SMQ020"] == 2,
        (df["SMQ020"] == 1) & (df["SMQ040"] == 3)
    ],
    ["Ideal", "Intermediate"],
    default="Poor"
)

# ── Alcohol Status ────────────────────────────────────────────────────────────
df["Alcohol_status"] = df["ALQ100"].map({
    1: "Alcohol_consumer",
    2: "Non_consumer"
})
df["Alcohol_status"] = pd.Categorical(
    df["Alcohol_status"],
    categories=["Non_consumer", "Alcohol_consumer"],
    ordered=True
)

# ── Physical Activity Category ────────────────────────────────────────────────
aerobic  = (df["PAD200"] == 1) | (df["PAD320"] == 1)
strength = (df["PAD440"] == 1) & (df["PAD460"] >= 8)

df["Physical_activity"] = np.select(
    [aerobic & strength, ~aerobic & (df["PAD440"] != 1)],
    ["Ideal", "Poor"],
    default="Intermediate"
)

# ── Select & Rename Final Columns ─────────────────────────────────────────────
# BUG FIX 3: DID040 (age at diagnosis) and DIAB_DUR were missing from the
#            final column list; added here alongside all original variables.
final = df[[
    "SEQN", "RIDAGEYR", "RIAGENDR", "RIDEXPRG", "RIDRETH1", "DMDEDUC2", "INDFMPIR",
    "BMXBMI", "SBP", "DBP", "LBXGH", "LBDLDL", "LBDHDL", "LBXTR",
    "LBXGLUSI", "LBXIN", "HOMA_IR", "HOMA_B", "LBDSGLSI",
    "LBXSCR", "eGFR", "PHAFSTHR",
    "DIQ010", "DIQ040Q", "DIQ050", "DIQ070", "DIAB_DUR",          # ← diabetes block
    "Smoking_category", "Alcohol_status", "Physical_activity"
]].rename(columns={
    "RIDAGEYR":   "Age",
    "RIAGENDR":   "Sex",
    "RIDEXPRG":   "Pregnancy",
    "RIDRETH1":   "Ethnicity",
    "DMDEDUC2":   "Education_level",
    "INDFMPIR":   "Family_PIR",
    "BMXBMI":     "BMI",
    "SBP":        "Systolic_BP",
    "DBP":        "Diastolic_BP",
    "LBXGH":      "HbA1c",
    "LBDLDL":     "LDL",
    "LBDHDL":     "HDL",
    "LBXTR":      "Triglycerides",
    "LBXGLUSI":   "Fasting_glucose",
    "LBDSGLSI":   "Glucose",
    "LBXIN":      "Fasting_insulin",
    "LBXSCR":     "Creatinine",
    "PHAFSTHR":   "Fasting_hours",
    "DIQ010":     "Diabetes",
    "DIQ050":      "Insulin_pill",
    "DIQ070":      "Diabetes_pill",
    "DIQ040Q":     "Age_at_diagnosis",         # ← renamed
    "DIAB_DUR":   "Diabetes_duration"         # ← new
})

# ── Recode Ethnicity ──────────────────────────────────────────────────────────
ethnicity_map = {
    1: "Mexican American",
    2: "Other Hispanic",
    3: "Non-Hispanic White",
    4: "Non-Hispanic Black",
    5: "Other Race"
}
final["Ethnicity"] = final["Ethnicity"].map(ethnicity_map)

# ── Export ────────────────────────────────────────────────────────────────────
final.to_csv("nhanes_data_csvm/nhanes_1999_2000_clean.csv", index=False)
print("✅ Saved:", final.shape, "rows")
print(final[["Diabetes", "Age_at_diagnosis", "Diabetes_duration"]].dropna(subset=["Diabetes_duration"]).head(10))
print(final.head())

## Cycle 2: NHANES 2001-2002

- Component files: suffix **`_B`**.
- eGFR: CKD-EPI **2021**. Physical activity: binary-item rule.
- Creatinine uses **`LBDSCR`** (not `LBXSCR`) - the one cycle where this differs; fasting glucose `LBXGLUSI`; HDL `LBDHDL`; alcohol **`ALD100`**.
- REVIEW: age at diagnosis from `DID040Q` (see Known issues #2).

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set the directory containing all .XPT files
BASE_DIR = Path(r"C:\Users\...\...\...\2001-2002")

# Load required files
demo     = pd.read_sas(BASE_DIR / "DEMO_B.XPT")
bmx      = pd.read_sas(BASE_DIR / "BMX_B.XPT")
bpx      = pd.read_sas(BASE_DIR / "BPX_B.XPT")
hdl      = pd.read_sas(BASE_DIR / "L13_B.XPT")
trig     = pd.read_sas(BASE_DIR / "L13AM_B.XPT")
hba1c    = pd.read_sas(BASE_DIR / "L10_B.XPT")
glu      = pd.read_sas(BASE_DIR / "L10AM_B.XPT")
creat    = pd.read_sas(BASE_DIR / "L40_B.XPT")
fast     = pd.read_sas(BASE_DIR / "PH_B.XPT")
diabetes = pd.read_sas(BASE_DIR / "DIQ_B.XPT")
smoking  = pd.read_sas(BASE_DIR / "SMQ_B.XPT")
alcohol  = pd.read_sas(BASE_DIR / "ALQ_B.XPT")
activity = pd.read_sas(BASE_DIR / "PAQ_B.XPT")

# ── Select Relevant Features ──────────────────────────────────────────────────
demo     = demo[["SEQN", "RIDAGEYR", "RIAGENDR", "RIDEXPRG", "RIDRETH1", "DMDEDUC2", "INDFMPIR"]]
bmx      = bmx[["SEQN", "BMXBMI"]]
bpx      = bpx[["SEQN", "BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4",
                        "BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]]
hba1c    = hba1c[["SEQN", "LBXGH"]]
trig     = trig[["SEQN", "LBDLDL", "LBXTR"]]
hdl      = hdl[["SEQN", "LBDHDL"]]
glu      = glu[["SEQN", "LBXGLUSI", "LBXIN"]]


creat    = creat[["SEQN", "LBDSCR", "LBDSGLSI"]]

fast     = fast[["SEQN", "PHAFSTHR"]]


diabetes = diabetes[["SEQN", "DIQ010", "DID040Q", "DIQ050", "DIQ070"]]

smoking  = smoking[["SEQN", "SMQ020", "SMQ040"]]
alcohol  = alcohol[["SEQN", "ALD100"]]
activity = activity[["SEQN", "PAD200", "PAD320", "PAD440", "PAD460"]]

# ── Mean Blood Pressure ───────────────────────────────────────────────────────
bpx["SBP"] = bpx[["BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4"]].mean(axis=1, skipna=True)
bpx["DBP"] = bpx[["BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]].mean(axis=1, skipna=True)
 
# ──eGFR CKD-EPI (2021) ───────────────────────────────────────────────────────
def compute_egfr_2021(creatinine_mgdl, age, gender):
    age             = pd.to_numeric(age,             errors='coerce')
    gender          = pd.to_numeric(gender,          errors='coerce')
    creatinine_mgdl = pd.to_numeric(creatinine_mgdl, errors='coerce')

    k         = np.where(gender == 1, 0.9,    0.7)
    alpha     = np.where(gender == 1, -0.302, -0.241)
    sex_coeff = np.where(gender == 1, 1.0,    1.012)

    min_scr = np.minimum(creatinine_mgdl / k, 1)
    max_scr = np.maximum(creatinine_mgdl / k, 1)

    egfr = 142 * (min_scr ** alpha) * (max_scr ** -1.200) * (0.9938 ** age) * sex_coeff
    return egfr

# ── HOMA-IR and HOMA-B ────────────────────────────────────────────────────────
def compute_homa(glu, ins, fasting_hours):
    glu  = pd.to_numeric(glu,           errors='coerce')
    ins  = pd.to_numeric(ins,           errors='coerce')
    
    fast = pd.to_numeric(fasting_hours, errors='coerce')

    if glu.median(skipna=True) > 30:   # Likely mg/dL → convert to mmol/L
        glu = glu / 18.018

    mask_fasting = fast >= 8           # ← fixed: was `fasting_hours >= 8`
    homa_ir = np.where(mask_fasting, (glu * ins) / 22.5, np.nan)
    homa_b  = np.where(mask_fasting & (glu > 3.5), (20 * ins) / (glu - 3.5), np.nan)
    return homa_ir, homa_b

# ── Merge all DataFrames on SEQN ──────────────────────────────────────────────
df = (
    demo
    .merge(bmx,                         on="SEQN", how="left")
    .merge(bpx[["SEQN", "SBP", "DBP"]], on="SEQN", how="left")
    .merge(hba1c,                       on="SEQN", how="left")
    .merge(trig,                        on="SEQN", how="left")
    .merge(glu,                         on="SEQN", how="left")
    .merge(hdl,                         on="SEQN", how="left")
    .merge(creat,                       on="SEQN", how="left")
    .merge(fast,                        on="SEQN", how="left")
    .merge(diabetes,                    on="SEQN", how="left")
    .merge(smoking,                     on="SEQN", how="left")
    .merge(alcohol,                     on="SEQN", how="left")
    .merge(activity,                    on="SEQN", how="left")
)

# ── Derived Variables ─────────────────────────────────────────────────────────
# BUG FIX 2 (continued): eGFR call updated to use corrected `LBXSCR` column
df["eGFR"]                  = compute_egfr_2021(df["LBDSCR"], df["RIDAGEYR"], df["RIAGENDR"])
df["HOMA_IR"], df["HOMA_B"] = compute_homa(df["LBXGLUSI"], df["LBXIN"], df["PHAFSTHR"])

# ── Diabetes Duration Workflow ────────────────────────────────────────────────
# Step 1 – Identify participants with confirmed diabetes (DIQ010 = 1)
has_diabetes = df["DIQ010"] == 1

# Step 2 – Extract DID040 (age at diagnosis); sentinel values
#           (777 = Refused, 999 = Don't know) and implausible ages → NaN
age_at_dx = pd.to_numeric(df["DID040Q"], errors='coerce')
age_at_dx = age_at_dx.where(age_at_dx < 120)

# Step 3 – Current age as numeric
current_age = pd.to_numeric(df["RIDAGEYR"], errors='coerce')

# Step 4 – DIAB_DUR: only populated for confirmed diabetics with a valid diagnosis age
df["DIAB_DUR"] = np.where(
    has_diabetes & age_at_dx.notna(),
    (current_age - age_at_dx).clip(lower=0),  # floor at 0 to prevent negatives
    np.nan
)

# ── Smoking Category ──────────────────────────────────────────────────────────
df["Smoking_category"] = np.select(
    [
        df["SMQ020"] == 2,
        (df["SMQ020"] == 1) & (df["SMQ040"] == 3)
    ],
    ["Ideal", "Intermediate"],
    default="Poor"
)

# ── Alcohol Status ────────────────────────────────────────────────────────────
df["Alcohol_status"] = df["ALD100"].map({
    1: "Alcohol_consumer",
    2: "Non_consumer"
})
df["Alcohol_status"] = pd.Categorical(
    df["Alcohol_status"],
    categories=["Non_consumer", "Alcohol_consumer"],
    ordered=True
)

# ── Physical Activity Category ────────────────────────────────────────────────
aerobic  = (df["PAD200"] == 1) | (df["PAD320"] == 1)
strength = (df["PAD440"] == 1) & (df["PAD460"] >= 8)

df["Physical_activity"] = np.select(
    [aerobic & strength, ~aerobic & (df["PAD440"] != 1)],
    ["Ideal", "Poor"],
    default="Intermediate"
)

# ── Select & Rename Final Columns ─────────────────────────────────────────────
# BUG FIX 4: DID040 (age at diagnosis) and DIAB_DUR were missing from the
#            final column list; added here alongside all original variables.
# Note: column order also normalised (RIDRETH1 moved back after RIDEXPRG to
#       match the consistent ordering used across all other cycle scripts).
final = df[[
    "SEQN", "RIDAGEYR", "RIAGENDR", "RIDEXPRG", "RIDRETH1", "DMDEDUC2", "INDFMPIR",
    "BMXBMI", "SBP", "DBP", "LBXGH", "LBDLDL", "LBDHDL", "LBXTR",
    "LBXGLUSI", "LBXIN", "HOMA_IR", "HOMA_B", "LBDSGLSI",
    "LBDSCR", "eGFR", "PHAFSTHR",
    "DIQ010", "DID040Q", "DIQ050", "DIQ070", "DIAB_DUR",          # ← diabetes block
    "Smoking_category", "Alcohol_status", "Physical_activity"
]].rename(columns={
    "RIDAGEYR":  "Age",
    "RIAGENDR":  "Sex",
    "RIDEXPRG":  "Pregnancy",
    "RIDRETH1":  "Ethnicity",
    "DMDEDUC2":  "Education_level",
    "INDFMPIR":  "Family_PIR",
    "BMXBMI":    "BMI",
    "SBP":       "Systolic_BP",
    "DBP":       "Diastolic_BP",
    "LBXGH":     "HbA1c",
    "LBDLDL":    "LDL",
    "LBDHDL":    "HDL",
    "LBXTR":     "Triglycerides",
    "LBXGLUSI":  "Fasting_glucose",
    "LBXIN":     "Fasting_insulin",
    "LBDSCR":    "Creatinine",               # ← updated from LBDSCR
    "LBDSGLSI":   "Glucose",
    "PHAFSTHR":  "Fasting_hours",
    "DIQ010":    "Diabetes",
    "DIQ050":    "Insulin_pill",
    "DIQ070":    "Diabetes_pill",
    "DID040Q":    "Age_at_diagnosis",          # ← renamed
    "DIAB_DUR":  "Diabetes_duration"          # ← new
})

# ── Recode Ethnicity ──────────────────────────────────────────────────────────
ethnicity_map = {
    1: "Mexican American",
    2: "Other Hispanic",
    3: "Non-Hispanic White",
    4: "Non-Hispanic Black",
    5: "Other Race"
}
final["Ethnicity"] = final["Ethnicity"].map(ethnicity_map)

# ── Export ────────────────────────────────────────────────────────────────────
final.to_csv("nhanes_data_csvm/nhanes_2001_2002_clean.csv", index=False)
print("✅ Saved:", final.shape, "rows")
print(final[["Diabetes", "Age_at_diagnosis", "Diabetes_duration"]].dropna(subset=["Diabetes_duration"]).head(10))


## Cycle 3: NHANES 2003-2004

- Component files: suffix **`_C`**.
- **eGFR: CKD-EPI 2009** here (`compute_egfr_2009`) - every other cycle uses the 2021 equation, so this cycle's eGFR is on a different scale (see Known issues #1).
- Physical activity: binary-item rule. HDL **`LBXHDD`**; fasting glucose `LBDGLUSI`; alcohol `ALQ101`.
- REVIEW: age at diagnosis from `DID040Q`; this cell's own comment flags `DID040Q` as a qualifier variable (see Known issues #2).

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set the directory containing all .XPT files
BASE_DIR = Path(r"C:\Users\...\...\2003-2004")

# Load required files
demo     = pd.read_sas(BASE_DIR / "DEMO_C.XPT")
bmx      = pd.read_sas(BASE_DIR / "BMX_C.XPT")
bpx      = pd.read_sas(BASE_DIR / "BPX_C.XPT")
hdl      = pd.read_sas(BASE_DIR / "L13_C.XPT")
trig     = pd.read_sas(BASE_DIR / "L13AM_C.XPT")
hba1c    = pd.read_sas(BASE_DIR / "L10_C.XPT")
glu      = pd.read_sas(BASE_DIR / "L10AM_C.XPT")
creat    = pd.read_sas(BASE_DIR / "L40_C.XPT")
fast     = pd.read_sas(BASE_DIR / "PH_C.XPT")
diabetes = pd.read_sas(BASE_DIR / "DIQ_C.XPT")
smoking  = pd.read_sas(BASE_DIR / "SMQ_C.XPT")
alcohol  = pd.read_sas(BASE_DIR / "ALQ_C.XPT")
activity = pd.read_sas(BASE_DIR / "PAQ_C.XPT")

# ── Select Relevant Features ──────────────────────────────────────────────────
demo     = demo[["SEQN", "RIDAGEYR", "RIAGENDR", "RIDEXPRG", "RIDRETH1", "DMDEDUC2", "INDFMPIR"]]
bmx      = bmx[["SEQN", "BMXBMI"]]
bpx      = bpx[["SEQN", "BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4",
                        "BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]]
hba1c    = hba1c[["SEQN", "LBXGH"]]
trig     = trig[["SEQN", "LBDLDL", "LBXTR"]]
hdl      = hdl[["SEQN", "LBXHDD"]]
glu      = glu[["SEQN", "LBDGLUSI", "LBXIN"]]
creat    = creat[["SEQN", "LBXSCR", "LBDSGLSI" ]]
fast     = fast[["SEQN", "PHAFSTHR"]]

# BUG FIX 1: DID040Q is a qualifier/flag variable, NOT age at diagnosis.
#            The correct variable for age at diagnosis is DID040 (numeric).
diabetes = diabetes[["SEQN", "DIQ010", "DID040Q", "DIQ050", "DIQ070"]]

smoking  = smoking[["SEQN", "SMQ020", "SMQ040"]]
alcohol  = alcohol[["SEQN", "ALQ101"]]
activity = activity[["SEQN", "PAD200", "PAD320", "PAD440", "PAD460"]]

# ── Mean Blood Pressure ───────────────────────────────────────────────────────
bpx["SBP"] = bpx[["BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4"]].mean(axis=1, skipna=True)
bpx["DBP"] = bpx[["BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]].mean(axis=1, skipna=True)

# ── eGFR CKD-EPI (2009) ───────────────────────────────────────────────────────
def compute_egfr_2009(creatinine_mgdl, age, gender):
    creatinine_mgdl = pd.to_numeric(creatinine_mgdl, errors='coerce')
    age             = pd.to_numeric(age,             errors='coerce')
    gender          = pd.to_numeric(gender,          errors='coerce')

    k         = np.where(gender == 1, 0.9,    0.7)
    alpha     = np.where(gender == 1, -0.411, -0.329)
    sex_coeff = np.where(gender == 1, 1.0,    1.018)

    min_scr = np.minimum(creatinine_mgdl / k, 1)
    max_scr = np.maximum(creatinine_mgdl / k, 1)

    egfr = 141 * (min_scr ** alpha) * (max_scr ** -1.209) * (0.993 ** age) * sex_coeff
    return egfr

# ── HOMA-IR and HOMA-B ────────────────────────────────────────────────────────
def compute_homa(glu, ins, fasting_hours):
    glu  = pd.to_numeric(glu,           errors='coerce')
    ins  = pd.to_numeric(ins,           errors='coerce')
    fast = pd.to_numeric(fasting_hours, errors='coerce')

    if glu.median(skipna=True) > 30:   # Likely mg/dL → convert to mmol/L
        glu = glu / 18.018

    mask_fasting = fast >= 8           # ← fixed: was `fasting_hours >= 8`
    homa_ir = np.where(mask_fasting, (glu * ins) / 22.5, np.nan)
    homa_b  = np.where(mask_fasting & (glu > 3.5), (20 * ins) / (glu - 3.5), np.nan)
    return homa_ir, homa_b

# ── Merge all DataFrames on SEQN ──────────────────────────────────────────────
df = (
    demo
    .merge(bmx,                         on="SEQN", how="left")
    .merge(bpx[["SEQN", "SBP", "DBP"]], on="SEQN", how="left")
    .merge(hba1c,                       on="SEQN", how="left")
    .merge(trig,                        on="SEQN", how="left")
    .merge(glu,                         on="SEQN", how="left")
    .merge(hdl,                         on="SEQN", how="left")
    .merge(creat,                       on="SEQN", how="left")
    .merge(fast,                        on="SEQN", how="left")
    .merge(diabetes,                    on="SEQN", how="left")
    .merge(smoking,                     on="SEQN", how="left")
    .merge(alcohol,                     on="SEQN", how="left")
    .merge(activity,                    on="SEQN", how="left")
)

# ── Derived Variables ─────────────────────────────────────────────────────────
df["eGFR"]                  = compute_egfr_2009(df["LBXSCR"], df["RIDAGEYR"], df["RIAGENDR"])
df["HOMA_IR"], df["HOMA_B"] = compute_homa(df["LBDGLUSI"], df["LBXIN"], df["PHAFSTHR"])

# ── Diabetes Duration Workflow ────────────────────────────────────────────────
# Step 1 – Identify participants with confirmed diabetes (DIQ010 = 1)
has_diabetes = df["DIQ010"] == 1

# Step 2 – Extract DID040 (age at diagnosis); sentinel values
#           (777 = Refused, 999 = Don't know) and implausible ages → NaN
age_at_dx = pd.to_numeric(df["DID040Q"], errors='coerce')
age_at_dx = age_at_dx.where(age_at_dx < 120)

# Step 3 – Current age as numeric
current_age = pd.to_numeric(df["RIDAGEYR"], errors='coerce')

# Step 4 – DIAB_DUR: only populated for confirmed diabetics with a valid diagnosis age
df["DIAB_DUR"] = np.where(
    has_diabetes & age_at_dx.notna(),
    (current_age - age_at_dx).clip(lower=0),  # floor at 0 to prevent negatives
    np.nan
)

# ── Smoking Category ──────────────────────────────────────────────────────────
df["Smoking_category"] = np.select(
    [
        df["SMQ020"] == 2,
        (df["SMQ020"] == 1) & (df["SMQ040"] == 3)
    ],
    ["Ideal", "Intermediate"],
    default="Poor"
)

# ── Alcohol Status ────────────────────────────────────────────────────────────
df["Alcohol_status"] = df["ALQ101"].map({
    1: "Alcohol_consumer",
    2: "Non_consumer"
})
df["Alcohol_status"] = pd.Categorical(
    df["Alcohol_status"],
    categories=["Non_consumer", "Alcohol_consumer"],
    ordered=True
)

# ── Physical Activity Category ────────────────────────────────────────────────
aerobic  = (df["PAD200"] == 1) | (df["PAD320"] == 1)
strength = (df["PAD440"] == 1) & (df["PAD460"] >= 8)

df["Physical_activity"] = np.select(
    [aerobic & strength, ~aerobic & (df["PAD440"] != 1)],
    ["Ideal", "Poor"],
    default="Intermediate"
)

# ── Select & Rename Final Columns ─────────────────────────────────────────────
# BUG FIX 5: DID040Q (age at diagnosis) and DIAB_DUR were missing from the
#            final column list; added here alongside all original variables.
final = df[[
    "SEQN", "RIDAGEYR", "RIAGENDR", "RIDEXPRG", "RIDRETH1", "DMDEDUC2", "INDFMPIR",
    "BMXBMI", "SBP", "DBP", "LBXGH", "LBDLDL", "LBXHDD", "LBXTR",
    "LBDGLUSI", "LBXIN", "HOMA_IR", "HOMA_B", "LBDSGLSI",
    "LBXSCR", "eGFR", "PHAFSTHR",
    "DIQ010", "DID040Q","DIQ050", "DIQ070", "DIAB_DUR",          # ← diabetes block
    "Smoking_category", "Alcohol_status", "Physical_activity"
]].rename(columns={
    "RIDAGEYR":  "Age",
    "RIAGENDR":  "Sex",
    "RIDEXPRG":  "Pregnancy",
    "RIDRETH1":  "Ethnicity",
    "DMDEDUC2":  "Education_level",
    "INDFMPIR":  "Family_PIR",
    "BMXBMI":    "BMI",
    "SBP":       "Systolic_BP",
    "DBP":       "Diastolic_BP",
    "LBXGH":     "HbA1c",
    "LBDLDL":    "LDL",
    "LBXHDD":    "HDL",
    "LBXTR":     "Triglycerides",
    "LBDGLUSI":  "Fasting_glucose",
    "LBXIN":     "Fasting_insulin",
    "LBXSCR":    "Creatinine",
    "LBDSGLSI":   "Glucose",
    "PHAFSTHR":  "Fasting_hours",
    "DIQ010":    "Diabetes",
    "DIQ050":    "Insulin_pill",
    "DIQ070":     "Diabetes_pill",
    "DID040Q":    "Age_at_diagnosis",          # ← renamed
    "DIAB_DUR":  "Diabetes_duration"          # ← new
})

# ── Recode Ethnicity ──────────────────────────────────────────────────────────
ethnicity_map = {
    1: "Mexican American",
    2: "Other Hispanic",
    3: "Non-Hispanic White",
    4: "Non-Hispanic Black",
    5: "Other Race"
}
final["Ethnicity"] = final["Ethnicity"].map(ethnicity_map)

# ── Export ────────────────────────────────────────────────────────────────────
final.to_csv("nhanes_data_csvm/nhanes_2003_2004_clean.csv", index=False)
print("✅ Saved:", final.shape, "rows")
print(final[["Diabetes", "Age_at_diagnosis", "Diabetes_duration"]].dropna(subset=["Diabetes_duration"]).head(10))

## Cycle 4: NHANES 2005-2006

- Component files: suffix **`_D`**; lab files switch to the modern names (`HDL_D`, `TRIGLY_D`, `GHB_D`, `GLU_D`, `BIOPRO_D`, `FASTQX_D`).
- eGFR: CKD-EPI 2021. Physical activity: binary-item rule (last cycle using it).
- First cycle with **OGTT** merged, adding `Two_hour_glucose` (`LBDGLTSI`).
- Age at diagnosis from **`DID040`** (the numeric variable); diabetes-pill variable is `DID070`; HDL `LBDHDD`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set the directory containing all .XPT files
BASE_DIR = Path(r"C:\Users\...\...\2005-2006")

# Load required files
demo     = pd.read_sas(BASE_DIR / "DEMO_D.XPT")
bmx      = pd.read_sas(BASE_DIR / "BMX_D.XPT")
bpx      = pd.read_sas(BASE_DIR / "BPX_D.XPT")
hdl      = pd.read_sas(BASE_DIR / "HDL_D.XPT")
trig     = pd.read_sas(BASE_DIR / "TRIGLY_D.XPT")
hba1c    = pd.read_sas(BASE_DIR / "GHB_D.XPT")
glu      = pd.read_sas(BASE_DIR / "GLU_D.XPT")
creat    = pd.read_sas(BASE_DIR / "BIOPRO_D.XPT")
fast     = pd.read_sas(BASE_DIR / "FASTQX_D.XPT")
ogtt     = pd.read_sas(BASE_DIR / "OGTT_D.XPT")
diabetes = pd.read_sas(BASE_DIR / "DIQ_D.XPT")
smoking  = pd.read_sas(BASE_DIR / "SMQ_D.XPT")
alcohol  = pd.read_sas(BASE_DIR / "ALQ_D.XPT")
activity = pd.read_sas(BASE_DIR / "PAQ_D.XPT")

# ── Select Relevant Features ──────────────────────────────────────────────────
demo     = demo[["SEQN", "RIDAGEYR", "RIAGENDR", "RIDEXPRG", "RIDRETH1", "DMDEDUC2", "INDFMPIR"]]
bmx      = bmx[["SEQN", "BMXBMI"]]
bpx      = bpx[["SEQN", "BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4",
                        "BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]]
hba1c    = hba1c[["SEQN", "LBXGH"]]
trig     = trig[["SEQN", "LBDLDL", "LBXTR"]]
hdl      = hdl[["SEQN", "LBDHDD"]]
glu      = glu[["SEQN", "LBDGLUSI", "LBXIN"]]
creat    = creat[["SEQN", "LBXSCR", "LBDSGLSI"]]
fast     = fast[["SEQN", "PHAFSTHR"]]
ogtt     = ogtt[["SEQN", "LBDGLTSI"]]
# NOTE: DID040 was already correctly selected in this cycle (no flag/qualifier mix-up)
diabetes = diabetes[["SEQN", "DIQ010", "DID040", "DIQ050", "DID070"]]
smoking  = smoking[["SEQN", "SMQ020", "SMQ040"]]
alcohol  = alcohol[["SEQN", "ALQ101"]]
activity = activity[["SEQN", "PAD200", "PAD320", "PAD440", "PAD460"]]

# ── Mean Blood Pressure ───────────────────────────────────────────────────────
bpx["SBP"] = bpx[["BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4"]].mean(axis=1, skipna=True)
bpx["DBP"] = bpx[["BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]].mean(axis=1, skipna=True)

# ── eGFR CKD-EPI (2021) ───────────────────────────────────────────────────────
def compute_egfr_2021(creatinine_mgdl, age, gender):
    age             = pd.to_numeric(age,             errors='coerce')
    gender          = pd.to_numeric(gender,          errors='coerce')
    creatinine_mgdl = pd.to_numeric(creatinine_mgdl, errors='coerce')

    k         = np.where(gender == 1, 0.9,    0.7)
    alpha     = np.where(gender == 1, -0.302, -0.241)
    sex_coeff = np.where(gender == 1, 1.0,    1.012)

    min_scr = np.minimum(creatinine_mgdl / k, 1)
    max_scr = np.maximum(creatinine_mgdl / k, 1)

    egfr = 142 * (min_scr ** alpha) * (max_scr ** -1.200) * (0.9938 ** age) * sex_coeff
    return egfr

# ── HOMA-IR and HOMA-B ────────────────────────────────────────────────────────
def compute_homa(glu, ins, fasting_hours):
    glu  = pd.to_numeric(glu,           errors='coerce')
    ins  = pd.to_numeric(ins,           errors='coerce')
    
    fast = pd.to_numeric(fasting_hours, errors='coerce')

    if glu.median(skipna=True) > 30:   # Likely mg/dL → convert to mmol/L
        glu = glu / 18.018

    mask_fasting = fast >= 8           # ← fixed: was `fasting_hours >= 8`
    homa_ir = np.where(mask_fasting, (glu * ins) / 22.5, np.nan)
    homa_b  = np.where(mask_fasting & (glu > 3.5), (20 * ins) / (glu - 3.5), np.nan)
    return homa_ir, homa_b

# ── Merge all DataFrames on SEQN ──────────────────────────────────────────────
df = (
    demo
    .merge(bmx,                         on="SEQN", how="left")
    .merge(bpx[["SEQN", "SBP", "DBP"]], on="SEQN", how="left")
    .merge(hba1c,                       on="SEQN", how="left")
    .merge(trig,                        on="SEQN", how="left")
    .merge(glu,                         on="SEQN", how="left")
    .merge(hdl,                         on="SEQN", how="left")
    .merge(creat,                       on="SEQN", how="left")
    .merge(fast,                        on="SEQN", how="left")
    .merge(ogtt,                        on="SEQN", how="left")
    .merge(diabetes,                    on="SEQN", how="left")
    .merge(smoking,                     on="SEQN", how="left")
    .merge(alcohol,                     on="SEQN", how="left")
    .merge(activity,                    on="SEQN", how="left")
)

# ── Derived Variables ─────────────────────────────────────────────────────────
df["eGFR"]                  = compute_egfr_2021(df["LBXSCR"], df["RIDAGEYR"], df["RIAGENDR"])
df["HOMA_IR"], df["HOMA_B"] = compute_homa(df["LBDGLUSI"], df["LBXIN"], df["PHAFSTHR"])

# ── Diabetes Duration Workflow ────────────────────────────────────────────────
# Step 1 – Identify participants with confirmed diabetes (DIQ010 = 1)
has_diabetes = df["DIQ010"] == 1

# Step 2 – Extract DID040 (age at diagnosis); sentinel values
#           (777 = Refused, 999 = Don't know) and implausible ages → NaN
age_at_dx = pd.to_numeric(df["DID040"], errors='coerce')
age_at_dx = age_at_dx.where(age_at_dx < 120)

# Step 3 – Current age as numeric
current_age = pd.to_numeric(df["RIDAGEYR"], errors='coerce')

# Step 4 – DIAB_DUR: only populated for confirmed diabetics with a valid diagnosis age
df["DIAB_DUR"] = np.where(
    has_diabetes & age_at_dx.notna(),
    (current_age - age_at_dx).clip(lower=0),  # floor at 0 to prevent negatives
    np.nan
)

# ── Smoking Category ──────────────────────────────────────────────────────────
df["Smoking_category"] = np.select(
    [
        df["SMQ020"] == 2,
        (df["SMQ020"] == 1) & (df["SMQ040"] == 3)
    ],
    ["Ideal", "Intermediate"],
    default="Poor"
)

# ── Alcohol Status ────────────────────────────────────────────────────────────
df["Alcohol_status"] = df["ALQ101"].map({
    1: "Alcohol_consumer",
    2: "Non_consumer"
})
df["Alcohol_status"] = pd.Categorical(
    df["Alcohol_status"],
    categories=["Non_consumer", "Alcohol_consumer"],
    ordered=True
)

# ── Physical Activity Category ────────────────────────────────────────────────
aerobic  = (df["PAD200"] == 1) | (df["PAD320"] == 1)
strength = (df["PAD440"] == 1) & (df["PAD460"] >= 8)

df["Physical_activity"] = np.select(
    [aerobic & strength, ~aerobic & (df["PAD440"] != 1)],
    ["Ideal", "Poor"],
    default="Intermediate"
)

# ── Select & Rename Final Columns ─────────────────────────────────────────────

final = df[[
    "SEQN", "RIDAGEYR", "RIAGENDR", "RIDEXPRG", "RIDRETH1", "DMDEDUC2", "INDFMPIR",
    "BMXBMI", "SBP", "DBP", "LBXGH", "LBDLDL", "LBDHDD", "LBXTR",
    "LBDGLUSI", "LBXIN", "HOMA_IR", "HOMA_B", "LBDSGLSI", "LBDGLTSI",
    "LBXSCR", "eGFR", "PHAFSTHR",
    "DIQ010", "DID040", "DIQ050", "DID070", "DIAB_DUR",          # ← diabetes block
    "Smoking_category", "Alcohol_status", "Physical_activity"
]].rename(columns={
    "RIDAGEYR":  "Age",
    "RIAGENDR":  "Sex",
    "RIDEXPRG":  "Pregnancy",
    "RIDRETH1":  "Ethnicity",
    "DMDEDUC2":  "Education_level",
    "INDFMPIR":  "Family_PIR",
    "BMXBMI":    "BMI",
    "SBP":       "Systolic_BP",
    "DBP":       "Diastolic_BP",
    "LBXGH":     "HbA1c",
    "LBDLDL":    "LDL",
    "LBDHDD":    "HDL",
    "LBXTR":     "Triglycerides",
    "LBDGLUSI":  "Fasting_glucose",
    "LBXIN":     "Fasting_insulin",
    "LBDSGLSI":  "Glucose",
    "LBDGLTSI":  "Two_hour_glucose",
    "LBXSCR":    "Creatinine",
    "PHAFSTHR":  "Fasting_hours",
    "DIQ010":    "Diabetes",
    "DIQ050":    "Insulin_pill",
    "DID070":    "Diabetes_pill",
    "DID040":    "Age_at_diagnosis",          # ← added
    "DIAB_DUR":  "Diabetes_duration"          # ← new
})

# ── Recode Ethnicity ──────────────────────────────────────────────────────────
ethnicity_map = {
    1: "Mexican American",
    2: "Other Hispanic",
    3: "Non-Hispanic White",
    4: "Non-Hispanic Black",
    5: "Other Race"
}
final["Ethnicity"] = final["Ethnicity"].map(ethnicity_map)

# ── Export ────────────────────────────────────────────────────────────────────
final.to_csv("nhanes_data_csvm/nhanes_2005_2006_clean.csv", index=False)
print("✅ Saved:", final.shape, "rows")
print(final[["Diabetes", "Age_at_diagnosis", "Diabetes_duration"]].dropna(subset=["Diabetes_duration"]).head(10))

## Cycle 5: NHANES 2007-2008

- Component files: suffix **`_E`**.
- **Physical activity switches to the MVPA-minutes method** (`PAQ610.../PAD615...`): days x minutes per domain, vigorous weighted double, Ideal at >= 150 min/week.
- Sentinel codes are cleaned first (7/9 for day counts, 7777/9999 for minutes); without this, refusals would inflate activity totals.
- eGFR: CKD-EPI 2021. OGTT included. Age at diagnosis `DID040`; diabetes-pill `DID070`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set the directory containing all .XPT files
BASE_DIR = Path(r"C:\Users\...\...\2007-2008")

# Load required files
demo     = pd.read_sas(BASE_DIR / "DEMO_E.XPT")
bmx      = pd.read_sas(BASE_DIR / "BMX_E.XPT")
bpx      = pd.read_sas(BASE_DIR / "BPX_E.XPT")
hdl      = pd.read_sas(BASE_DIR / "HDL_E.XPT")
trig     = pd.read_sas(BASE_DIR / "TRIGLY_E.XPT")
hba1c    = pd.read_sas(BASE_DIR / "GHB_E.XPT")
glu      = pd.read_sas(BASE_DIR / "GLU_E.XPT")
creat    = pd.read_sas(BASE_DIR / "BIOPRO_E.XPT")
fast     = pd.read_sas(BASE_DIR / "FASTQX_E.XPT")
ogtt     = pd.read_sas(BASE_DIR / "OGTT_E.XPT")
diabetes = pd.read_sas(BASE_DIR / "DIQ_E.XPT")
smoking  = pd.read_sas(BASE_DIR / "SMQ_E.XPT")
alcohol  = pd.read_sas(BASE_DIR / "ALQ_E.XPT")
activity = pd.read_sas(BASE_DIR / "PAQ_E.XPT")

# ── Select Relevant Features ──────────────────────────────────────────────────
demo     = demo[["SEQN", "RIDAGEYR", "RIAGENDR", "RIDEXPRG", "RIDRETH1", "DMDEDUC2", "INDFMPIR"]]
bmx      = bmx[["SEQN", "BMXBMI"]]
bpx      = bpx[["SEQN", "BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4",
                        "BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]]
hba1c    = hba1c[["SEQN", "LBXGH"]]
trig     = trig[["SEQN", "LBDLDL", "LBXTR"]]
hdl      = hdl[["SEQN", "LBDHDD"]]
glu      = glu[["SEQN", "LBDGLUSI", "LBXIN"]]
creat    = creat[["SEQN", "LBXSCR", "LBDSGLSI"]]
fast     = fast[["SEQN", "PHAFSTHR"]]
ogtt     = ogtt[["SEQN", "LBDGLTSI"]]
# DID040 was already correctly selected in this cycle
diabetes = diabetes[["SEQN", "DIQ010", "DID040", "DIQ050", "DID070"]]
smoking  = smoking[["SEQN", "SMQ020", "SMQ040"]]
alcohol  = alcohol[["SEQN", "ALQ101"]]
activity = activity[["SEQN", "PAQ610", "PAD615", "PAQ655", "PAD660",
                             "PAQ625", "PAD630", "PAQ670", "PAD675"]]

# ── Mean Blood Pressure ───────────────────────────────────────────────────────
bpx["SBP"] = bpx[["BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4"]].mean(axis=1, skipna=True)
bpx["DBP"] = bpx[["BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]].mean(axis=1, skipna=True)

# ── eGFR CKD-EPI (2021) ───────────────────────────────────────────────────────
# BUG FIX 1: The original comment incorrectly labelled this as CKD-EPI (2009).
#            The coefficients used (142, -0.302/-0.241, -1.200, 0.9938, 1.012)
#            are the 2021 equation — the function name was correct all along.
def compute_egfr_2021(creatinine_mgdl, age, gender):
    age             = pd.to_numeric(age,             errors='coerce')
    gender          = pd.to_numeric(gender,          errors='coerce')
    creatinine_mgdl = pd.to_numeric(creatinine_mgdl, errors='coerce')

    k         = np.where(gender == 1, 0.9,    0.7)
    alpha     = np.where(gender == 1, -0.302, -0.241)
    sex_coeff = np.where(gender == 1, 1.0,    1.012)

    min_scr = np.minimum(creatinine_mgdl / k, 1)
    max_scr = np.maximum(creatinine_mgdl / k, 1)

    egfr = 142 * (min_scr ** alpha) * (max_scr ** -1.200) * (0.9938 ** age) * sex_coeff
    return egfr

# ── HOMA-IR and HOMA-B ────────────────────────────────────────────────────────
def compute_homa(glu, ins, fasting_hours):
    glu  = pd.to_numeric(glu,           errors='coerce')
    ins  = pd.to_numeric(ins,           errors='coerce')
    # BUG FIX 2: Original compared the raw `fasting_hours` parameter before
    #            numeric conversion. Use the converted `fast` variable instead.
    fast = pd.to_numeric(fasting_hours, errors='coerce')

    if glu.median(skipna=True) > 30:   # Likely mg/dL → convert to mmol/L
        glu = glu / 18.018

    mask_fasting = fast >= 8           # ← fixed: was `fasting_hours >= 8`
    homa_ir = np.where(mask_fasting, (glu * ins) / 22.5, np.nan)
    homa_b  = np.where(mask_fasting & (glu > 3.5), (20 * ins) / (glu - 3.5), np.nan)
    return homa_ir, homa_b

# ── Merge all DataFrames on SEQN ──────────────────────────────────────────────
df = (
    demo
    .merge(bmx,                         on="SEQN", how="left")
    .merge(bpx[["SEQN", "SBP", "DBP"]], on="SEQN", how="left")
    .merge(hba1c,                       on="SEQN", how="left")
    .merge(trig,                        on="SEQN", how="left")
    .merge(glu,                         on="SEQN", how="left")
    .merge(hdl,                         on="SEQN", how="left")
    .merge(creat,                       on="SEQN", how="left")
    .merge(fast,                        on="SEQN", how="left")
    .merge(ogtt,                        on="SEQN", how="left")
    .merge(diabetes,                    on="SEQN", how="left")
    .merge(smoking,                     on="SEQN", how="left")
    .merge(alcohol,                     on="SEQN", how="left")
    .merge(activity,                    on="SEQN", how="left")
)

# ── Derived Variables ─────────────────────────────────────────────────────────
df["eGFR"]                  = compute_egfr_2021(df["LBXSCR"], df["RIDAGEYR"], df["RIAGENDR"])
df["HOMA_IR"], df["HOMA_B"] = compute_homa(df["LBDGLUSI"], df["LBXIN"], df["PHAFSTHR"])

# ── Diabetes Duration Workflow ────────────────────────────────────────────────
# Step 1 – Identify participants with confirmed diabetes (DIQ010 = 1)
has_diabetes = df["DIQ010"] == 1

# Step 2 – Extract DID040 (age at diagnosis); sentinel values
#           (777 = Refused, 999 = Don't know) and implausible ages → NaN
age_at_dx = pd.to_numeric(df["DID040"], errors='coerce')
age_at_dx = age_at_dx.where(age_at_dx < 120)

# Step 3 – Current age as numeric
current_age = pd.to_numeric(df["RIDAGEYR"], errors='coerce')

# Step 4 – DIAB_DUR: only populated for confirmed diabetics with a valid diagnosis age
df["DIAB_DUR"] = np.where(
    has_diabetes & age_at_dx.notna(),
    (current_age - age_at_dx).clip(lower=0),  # floor at 0 to prevent negatives
    np.nan
)

# ── Smoking Category ──────────────────────────────────────────────────────────
df["Smoking_category"] = np.select(
    [
        df["SMQ020"] == 2,
        (df["SMQ020"] == 1) & (df["SMQ040"] == 3)
    ],
    ["Ideal", "Intermediate"],
    default="Poor"
)

# ── Alcohol Status ────────────────────────────────────────────────────────────
df["Alcohol_status"] = df["ALQ101"].map({
    1: "Alcohol_consumer",
    2: "Non_consumer"
})
df["Alcohol_status"] = pd.Categorical(
    df["Alcohol_status"],
    categories=["Non_consumer", "Alcohol_consumer"],
    ordered=True
)

# ── Physical Activity Category ────────────────────────────────────────────────
# BUG FIX 3: NHANES PAQ* day-count variables use 7 (Refused) / 9 (Don't know)
#            and PAD* minute variables use 7777 / 9999 as sentinel codes.
#            Without replacing these with NaN first, refused/unknown responses
#            silently inflate MVPA totals and misclassify participants.
#            All PAQ/PAD columns are sanitised before any arithmetic.

# Days per week: valid range 0–7; replace refusal/unknown codes with NaN
day_cols = ["PAQ610", "PAQ655", "PAQ625", "PAQ670"]
for col in day_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].where(df[col] <= 7)       # 7 = Refused, 9 = Don't know → NaN

# Minutes per day: valid range 10–840 per NHANES; replace sentinel codes with NaN
min_cols = ["PAD615", "PAD660", "PAD630", "PAD675"]
for col in min_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].where(df[col] < 7777)     # 7777 = Refused, 9999 = Don't know

# Vigorous minutes/week (work + recreational)
df["vigorous_min_week"] = (
    df["PAQ610"].fillna(0) * df["PAD615"].fillna(0) +
    df["PAQ655"].fillna(0) * df["PAD660"].fillna(0)
)

# Moderate minutes/week (walking/biking + recreational moderate)
df["moderate_min_week"] = (
    df["PAQ625"].fillna(0) * df["PAD630"].fillna(0) +
    df["PAQ670"].fillna(0) * df["PAD675"].fillna(0)
)

# MVPA equivalent (vigorous counts double per WHO/AHA guidelines)
df["mvpa_equivalent"] = df["moderate_min_week"] + 2 * df["vigorous_min_week"]

# Mark as Poor if all activity columns were missing (truly no data vs. zero activity)
all_missing = df[day_cols + min_cols].isna().all(axis=1)
df["mvpa_equivalent"] = df["mvpa_equivalent"].where(~all_missing, other=np.nan)

df["Physical_activity"] = np.select(
    [
        df["mvpa_equivalent"] >= 150,
        df["mvpa_equivalent"] > 0
    ],
    ["Ideal", "Intermediate"],
    default="Poor"
)

# ── Select & Rename Final Columns ─────────────────────────────────────────────
# BUG FIX 4 & 5: DID040 (age at diagnosis) and DIAB_DUR were missing from the
#                final column list despite DID040 being correctly loaded above.
final = df[[
    "SEQN", "RIDAGEYR", "RIAGENDR", "RIDEXPRG", "RIDRETH1", "DMDEDUC2", "INDFMPIR",
    "BMXBMI", "SBP", "DBP", "LBXGH", "LBDLDL", "LBDHDD", "LBXTR",
    "LBDGLUSI", "LBXIN", "HOMA_IR", "HOMA_B", "LBDSGLSI", "LBDGLTSI",
    "LBXSCR", "eGFR", "PHAFSTHR",
    "DIQ010", "DID040","DIQ050", "DID070", "DIAB_DUR",          # ← diabetes block
    "Smoking_category", "Alcohol_status", "Physical_activity"
]].rename(columns={
    "RIDAGEYR":  "Age",
    "RIAGENDR":  "Sex",
    "RIDEXPRG":  "Pregnancy",
    "RIDRETH1":  "Ethnicity",
    "DMDEDUC2":  "Education_level",
    "INDFMPIR":  "Family_PIR",
    "BMXBMI":    "BMI",
    "SBP":       "Systolic_BP",
    "DBP":       "Diastolic_BP",
    "LBXGH":     "HbA1c",
    "LBDLDL":    "LDL",
    "LBDHDD":    "HDL",
    "LBXTR":     "Triglycerides",
    "LBDGLUSI":  "Fasting_glucose",
    "LBXIN":     "Fasting_insulin",
    "LBDSGLSI":   "Glucose",
    "LBDGLTSI":   "Two_hour_glucose",
    "LBXSCR":    "Creatinine",
    "PHAFSTHR":  "Fasting_hours",
    "DIQ010":    "Diabetes",
    "DIQ050":     "Insulin_pill",
    "DID070":     "Diabetes_pill",
    "DID040":    "Age_at_diagnosis",          # ← added
    "DIAB_DUR":  "Diabetes_duration"          # ← new
})

# ── Recode Ethnicity ──────────────────────────────────────────────────────────
ethnicity_map = {
    1: "Mexican American",
    2: "Other Hispanic",
    3: "Non-Hispanic White",
    4: "Non-Hispanic Black",
    5: "Other Race"
}
final["Ethnicity"] = final["Ethnicity"].map(ethnicity_map)

# ── Export ────────────────────────────────────────────────────────────────────
final.to_csv("nhanes_data_csvm/nhanes_2007_2008_clean.csv", index=False)
print("✅ Saved:", final.shape, "rows")
print(final[["Diabetes", "Age_at_diagnosis", "Diabetes_duration"]].dropna(subset=["Diabetes_duration"]).head(10))

## Cycle 6: NHANES 2009-2010

- Component files: suffix **`_F`**.
- eGFR: CKD-EPI 2021. Physical activity: MVPA-minutes method. OGTT included.
- Age at diagnosis `DID040`; diabetes-pill reverts to `DIQ070`; alcohol `ALQ101`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set the directory containing all .XPT files
BASE_DIR = Path(r"C:\Users\...\...\2009-2010")

# Load required files
demo     = pd.read_sas(BASE_DIR / "DEMO_F.XPT")
bmx      = pd.read_sas(BASE_DIR / "BMX_F.XPT")
bpx      = pd.read_sas(BASE_DIR / "BPX_F.XPT")
hdl      = pd.read_sas(BASE_DIR / "HDL_F.XPT")
trig     = pd.read_sas(BASE_DIR / "TRIGLY_F.XPT")
hba1c    = pd.read_sas(BASE_DIR / "GHB_F.XPT")
glu      = pd.read_sas(BASE_DIR / "GLU_F.XPT")
creat    = pd.read_sas(BASE_DIR / "BIOPRO_F.XPT")
fast     = pd.read_sas(BASE_DIR / "FASTQX_F.XPT")
ogtt     = pd.read_sas(BASE_DIR / "OGTT_F.XPT")
diabetes = pd.read_sas(BASE_DIR / "DIQ_F.XPT")
smoking  = pd.read_sas(BASE_DIR / "SMQ_F.XPT")
alcohol  = pd.read_sas(BASE_DIR / "ALQ_F.XPT")
activity = pd.read_sas(BASE_DIR / "PAQ_F.XPT")

# ── Select Relevant Features ──────────────────────────────────────────────────
demo     = demo[["SEQN", "RIDAGEYR", "RIAGENDR", "RIDEXPRG", "RIDRETH1", "DMDEDUC2", "INDFMPIR"]]
bmx      = bmx[["SEQN", "BMXBMI"]]
bpx      = bpx[["SEQN", "BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4",
                        "BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]]
hba1c    = hba1c[["SEQN", "LBXGH"]]
trig     = trig[["SEQN", "LBDLDL", "LBXTR"]]
hdl      = hdl[["SEQN", "LBDHDD"]]
glu      = glu[["SEQN", "LBDGLUSI", "LBXIN"]]
creat    = creat[["SEQN", "LBXSCR", "LBDSGLSI"]]
fast     = fast[["SEQN", "PHAFSTHR"]]
ogtt     = ogtt[["SEQN", "LBDGLTSI"]]
# DID040 correctly selected in this cycle
diabetes = diabetes[["SEQN", "DIQ010", "DID040", "DIQ050", "DIQ070"]]
smoking  = smoking[["SEQN", "SMQ020", "SMQ040"]]
alcohol  = alcohol[["SEQN", "ALQ101"]]
activity = activity[["SEQN", "PAQ610", "PAD615", "PAQ655", "PAD660",
                             "PAQ625", "PAD630", "PAQ670", "PAD675"]]

# ── Mean Blood Pressure ───────────────────────────────────────────────────────
bpx["SBP"] = bpx[["BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4"]].mean(axis=1, skipna=True)
bpx["DBP"] = bpx[["BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]].mean(axis=1, skipna=True)

# ── eGFR CKD-EPI (2021) ───────────────────────────────────────────────────────
def compute_egfr_2021(creatinine_mgdl, age, gender):
    age             = pd.to_numeric(age,             errors='coerce')
    gender          = pd.to_numeric(gender,          errors='coerce')
    creatinine_mgdl = pd.to_numeric(creatinine_mgdl, errors='coerce')

    k         = np.where(gender == 1, 0.9,    0.7)
    alpha     = np.where(gender == 1, -0.302, -0.241)
    sex_coeff = np.where(gender == 1, 1.0,    1.012)

    min_scr = np.minimum(creatinine_mgdl / k, 1)
    max_scr = np.maximum(creatinine_mgdl / k, 1)

    egfr = 142 * (min_scr ** alpha) * (max_scr ** -1.200) * (0.9938 ** age) * sex_coeff
    return egfr

# ── HOMA-IR and HOMA-B ────────────────────────────────────────────────────────
def compute_homa(glu, ins, fasting_hours):
    glu  = pd.to_numeric(glu,           errors='coerce')
    ins  = pd.to_numeric(ins,           errors='coerce')
    # BUG FIX 2: Original compared the raw `fasting_hours` parameter before
    #            numeric conversion. Use the converted `fast` variable instead.
    fast = pd.to_numeric(fasting_hours, errors='coerce')

    if glu.median(skipna=True) > 30:   # Likely mg/dL → convert to mmol/L
        glu = glu / 18.018

    mask_fasting = fast >= 8           # ← fixed: was `fasting_hours >= 8`
    homa_ir = np.where(mask_fasting, (glu * ins) / 22.5, np.nan)
    homa_b  = np.where(mask_fasting & (glu > 3.5), (20 * ins) / (glu - 3.5), np.nan)
    return homa_ir, homa_b

# ── Merge all DataFrames on SEQN ──────────────────────────────────────────────
df = (
    demo
    .merge(bmx,                         on="SEQN", how="left")
    .merge(bpx[["SEQN", "SBP", "DBP"]], on="SEQN", how="left")
    .merge(hba1c,                       on="SEQN", how="left")
    .merge(trig,                        on="SEQN", how="left")
    .merge(glu,                         on="SEQN", how="left")
    .merge(hdl,                         on="SEQN", how="left")
    .merge(creat,                       on="SEQN", how="left")
    .merge(fast,                        on="SEQN", how="left")
    .merge(ogtt,                        on="SEQN", how="left")
    .merge(diabetes,                    on="SEQN", how="left")
    .merge(smoking,                     on="SEQN", how="left")
    .merge(alcohol,                     on="SEQN", how="left")
    .merge(activity,                    on="SEQN", how="left")
)

# ── Derived Variables ─────────────────────────────────────────────────────────
df["eGFR"]                  = compute_egfr_2021(df["LBXSCR"], df["RIDAGEYR"], df["RIAGENDR"])
df["HOMA_IR"], df["HOMA_B"] = compute_homa(df["LBDGLUSI"], df["LBXIN"], df["PHAFSTHR"])

# ── Diabetes Duration Workflow ────────────────────────────────────────────────
# Step 1 – Identify participants with confirmed diabetes (DIQ010 = 1)
has_diabetes = df["DIQ010"] == 1

# Step 2 – Extract DID040 (age at diagnosis); sentinel values
#           (777 = Refused, 999 = Don't know) and implausible ages → NaN
age_at_dx = pd.to_numeric(df["DID040"], errors='coerce')
age_at_dx = age_at_dx.where(age_at_dx < 120)

# Step 3 – Current age as numeric
current_age = pd.to_numeric(df["RIDAGEYR"], errors='coerce')

# Step 4 – DIAB_DUR: only populated for confirmed diabetics with a valid diagnosis age
df["DIAB_DUR"] = np.where(
    has_diabetes & age_at_dx.notna(),
    (current_age - age_at_dx).clip(lower=0),  # floor at 0 to prevent negatives
    np.nan
)

# ── Smoking Category ──────────────────────────────────────────────────────────
df["Smoking_category"] = np.select(
    [
        df["SMQ020"] == 2,
        (df["SMQ020"] == 1) & (df["SMQ040"] == 3)
    ],
    ["Ideal", "Intermediate"],
    default="Poor"
)

# ── Alcohol Status ────────────────────────────────────────────────────────────
df["Alcohol_status"] = df["ALQ101"].map({
    1: "Alcohol_consumer",
    2: "Non_consumer"
})
df["Alcohol_status"] = pd.Categorical(
    df["Alcohol_status"],
    categories=["Non_consumer", "Alcohol_consumer"],
    ordered=True
)

# ── Physical Activity Category ────────────────────────────────────────────────

# Days per week: valid range 0–7; replace refusal/unknown codes with NaN
day_cols = ["PAQ610", "PAQ655", "PAQ625", "PAQ670"]
for col in day_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].where(df[col] <= 7)       # 7 = Refused, 9 = Don't know → NaN

# Minutes per day: replace sentinel codes with NaN
min_cols = ["PAD615", "PAD660", "PAD630", "PAD675"]
for col in min_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].where(df[col] < 7777)     # 7777 = Refused, 9999 = Don't know

# Vigorous minutes/week (work + recreational)
df["vigorous_min_week"] = (
    df["PAQ610"].fillna(0) * df["PAD615"].fillna(0) +
    df["PAQ655"].fillna(0) * df["PAD660"].fillna(0)
)

# Moderate minutes/week (walking/biking + recreational moderate)
df["moderate_min_week"] = (
    df["PAQ625"].fillna(0) * df["PAD630"].fillna(0) +
    df["PAQ670"].fillna(0) * df["PAD675"].fillna(0)
)

# MVPA equivalent (vigorous counts double per WHO/AHA guidelines)
df["mvpa_equivalent"] = df["moderate_min_week"] + 2 * df["vigorous_min_week"]

# Mark as NaN if all activity columns were missing (no data vs. zero activity)
all_missing = df[day_cols + min_cols].isna().all(axis=1)
df["mvpa_equivalent"] = df["mvpa_equivalent"].where(~all_missing, other=np.nan)

df["Physical_activity"] = np.select(
    [
        df["mvpa_equivalent"] >= 150,
        df["mvpa_equivalent"] > 0
    ],
    ["Ideal", "Intermediate"],
    default="Poor"
)

# ── Select & Rename Final Columns ─────────────────────────────────────────────

final = df[[
    "SEQN", "RIDAGEYR", "RIAGENDR", "RIDEXPRG", "RIDRETH1", "DMDEDUC2", "INDFMPIR",
    "BMXBMI", "SBP", "DBP", "LBXGH", "LBDLDL", "LBDHDD", "LBXTR",
    "LBDGLUSI", "LBXIN", "HOMA_IR", "HOMA_B", "LBDSGLSI", "LBDGLTSI",
    "LBXSCR", "eGFR", "PHAFSTHR",
    "DIQ010", "DID040","DIQ050", "DIQ070", "DIAB_DUR",          # ← diabetes block
    "Smoking_category", "Alcohol_status", "Physical_activity"
]].rename(columns={
    "RIDAGEYR":  "Age",
    "RIAGENDR":  "Sex",
    "RIDEXPRG":  "Pregnancy",
    "RIDRETH1":  "Ethnicity",
    "DMDEDUC2":  "Education_level",
    "INDFMPIR":  "Family_PIR",
    "BMXBMI":    "BMI",
    "SBP":       "Systolic_BP",
    "DBP":       "Diastolic_BP",
    "LBXGH":     "HbA1c",
    "LBDLDL":    "LDL",
    "LBDHDD":    "HDL",
    "LBXTR":     "Triglycerides",
    "LBDGLUSI":  "Fasting_glucose",
    "LBXIN":     "Fasting_insulin",
    "LBDSGLSI":   "Glucose",
    "LBDGLTSI":   "Two_hour_glucose",
    "LBXSCR":    "Creatinine",
    "PHAFSTHR":  "Fasting_hours",
    "DIQ010":    "Diabetes",
    "DIQ050":    "Insulin_pill",
    "DIQ070":     "Diabetes_pill",
    "DID040":    "Age_at_diagnosis",          # ← added
    "DIAB_DUR":  "Diabetes_duration"          # ← new
})

# ── Recode Ethnicity ──────────────────────────────────────────────────────────
ethnicity_map = {
    1: "Mexican American",
    2: "Other Hispanic",
    3: "Non-Hispanic White",
    4: "Non-Hispanic Black",
    5: "Other Race"
}
final["Ethnicity"] = final["Ethnicity"].map(ethnicity_map)

# ── Export ────────────────────────────────────────────────────────────────────
final.to_csv("nhanes_data_csvm/nhanes_2009_2010_clean.csv", index=False)
print("✅ Saved:", final.shape, "rows")
print(final[["Diabetes", "Age_at_diagnosis", "Diabetes_duration"]].dropna(subset=["Diabetes_duration"]).head(10))

## Cycle 7: NHANES 2011-2012

- Component files: suffix **`_G`**.
- Structurally identical to 2009-2010: eGFR 2021, MVPA method, OGTT merged, `DID040`, `DIQ070`, `ALQ101`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set the directory containing all .XPT files
BASE_DIR = Path(r"C:\Users\...\...2011-2012")

# Load required files
demo     = pd.read_sas(BASE_DIR / "DEMO_G.XPT")
bmx      = pd.read_sas(BASE_DIR / "BMX_G.XPT")
bpx      = pd.read_sas(BASE_DIR / "BPX_G.XPT")
hdl      = pd.read_sas(BASE_DIR / "HDL_G.XPT")
trig     = pd.read_sas(BASE_DIR / "TRIGLY_G.XPT")
hba1c    = pd.read_sas(BASE_DIR / "GHB_G.XPT")
glu      = pd.read_sas(BASE_DIR / "GLU_G.XPT")
creat    = pd.read_sas(BASE_DIR / "BIOPRO_G.XPT")
fast     = pd.read_sas(BASE_DIR / "FASTQX_G.XPT")
ogtt     = pd.read_sas(BASE_DIR / "OGTT_G.XPT")
diabetes = pd.read_sas(BASE_DIR / "DIQ_G.XPT")
smoking  = pd.read_sas(BASE_DIR / "SMQ_G.XPT")
alcohol  = pd.read_sas(BASE_DIR / "ALQ_G.XPT")
activity = pd.read_sas(BASE_DIR / "PAQ_G.XPT")

# ── Select Relevant Features ──────────────────────────────────────────────────
demo     = demo[["SEQN", "RIDAGEYR", "RIAGENDR", "RIDEXPRG", "RIDRETH1", "DMDEDUC2", "INDFMPIR"]]
bmx      = bmx[["SEQN", "BMXBMI"]]
bpx      = bpx[["SEQN", "BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4",
                        "BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]]
hba1c    = hba1c[["SEQN", "LBXGH"]]
trig     = trig[["SEQN", "LBDLDL", "LBXTR"]]
hdl      = hdl[["SEQN", "LBDHDD"]]
glu      = glu[["SEQN", "LBDGLUSI", "LBXIN"]]
creat    = creat[["SEQN", "LBXSCR", "LBDSGLSI"]]
fast     = fast[["SEQN", "PHAFSTHR"]]
ogtt     = ogtt[["SEQN", "LBDGLTSI"]]

# DID040 correctly selected in this cycle
diabetes = diabetes[["SEQN", "DIQ010", "DID040", "DIQ050", "DIQ070"]]
smoking  = smoking[["SEQN", "SMQ020", "SMQ040"]]
alcohol  = alcohol[["SEQN", "ALQ101"]]
activity = activity[["SEQN", "PAQ610", "PAD615", "PAQ655", "PAD660",
                             "PAQ625", "PAD630", "PAQ670", "PAD675"]]

# ── Mean Blood Pressure ───────────────────────────────────────────────────────
bpx["SBP"] = bpx[["BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4"]].mean(axis=1, skipna=True)
bpx["DBP"] = bpx[["BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]].mean(axis=1, skipna=True)

# ── eGFR CKD-EPI (2021) ───────────────────────────────────────────────────────
def compute_egfr_2021(creatinine_mgdl, age, gender):
    age             = pd.to_numeric(age,             errors='coerce')
    gender          = pd.to_numeric(gender,          errors='coerce')
    creatinine_mgdl = pd.to_numeric(creatinine_mgdl, errors='coerce')

    k         = np.where(gender == 1, 0.9,    0.7)
    alpha     = np.where(gender == 1, -0.302, -0.241)
    sex_coeff = np.where(gender == 1, 1.0,    1.012)

    min_scr = np.minimum(creatinine_mgdl / k, 1)
    max_scr = np.maximum(creatinine_mgdl / k, 1)

    egfr = 142 * (min_scr ** alpha) * (max_scr ** -1.200) * (0.9938 ** age) * sex_coeff
    return egfr

# ── HOMA-IR and HOMA-B ────────────────────────────────────────────────────────
def compute_homa(glu, ins, fasting_hours):
    glu  = pd.to_numeric(glu,           errors='coerce')
    ins  = pd.to_numeric(ins,           errors='coerce')
    # BUG FIX 1: Original compared the raw `fasting_hours` parameter before
    #            numeric conversion. Use the converted `fast` variable instead.
    fast = pd.to_numeric(fasting_hours, errors='coerce')

    if glu.median(skipna=True) > 30:   # Likely mg/dL → convert to mmol/L
        glu = glu / 18.018

    mask_fasting = fast >= 8           # ← fixed: was `fasting_hours >= 8`
    homa_ir = np.where(mask_fasting, (glu * ins) / 22.5, np.nan)
    homa_b  = np.where(mask_fasting & (glu > 3.5), (20 * ins) / (glu - 3.5), np.nan)
    return homa_ir, homa_b

# ── Merge all DataFrames on SEQN ──────────────────────────────────────────────
df = (
    demo
    .merge(bmx,                         on="SEQN", how="left")
    .merge(bpx[["SEQN", "SBP", "DBP"]], on="SEQN", how="left")
    .merge(hba1c,                       on="SEQN", how="left")
    .merge(trig,                        on="SEQN", how="left")
    .merge(glu,                         on="SEQN", how="left")
    .merge(hdl,                         on="SEQN", how="left")
    .merge(creat,                       on="SEQN", how="left")
    .merge(fast,                        on="SEQN", how="left")
    .merge(ogtt,                        on="SEQN", how="left")
    .merge(diabetes,                    on="SEQN", how="left")
    .merge(smoking,                     on="SEQN", how="left")
    .merge(alcohol,                     on="SEQN", how="left")
    .merge(activity,                    on="SEQN", how="left")
)

# ── Derived Variables ─────────────────────────────────────────────────────────
df["eGFR"]                  = compute_egfr_2021(df["LBXSCR"], df["RIDAGEYR"], df["RIAGENDR"])
df["HOMA_IR"], df["HOMA_B"] = compute_homa(df["LBDGLUSI"], df["LBXIN"], df["PHAFSTHR"])

# ── Diabetes Duration Workflow ────────────────────────────────────────────────
# Step 1 – Identify participants with confirmed diabetes (DIQ010 = 1)
has_diabetes = df["DIQ010"] == 1

# Step 2 – Extract DID040 (age at diagnosis); sentinel values
#           (777 = Refused, 999 = Don't know) and implausible ages → NaN
age_at_dx = pd.to_numeric(df["DID040"], errors='coerce')
age_at_dx = age_at_dx.where(age_at_dx < 120)

# Step 3 – Current age as numeric
current_age = pd.to_numeric(df["RIDAGEYR"], errors='coerce')

# Step 4 – DIAB_DUR: only populated for confirmed diabetics with a valid diagnosis age
df["DIAB_DUR"] = np.where(
    has_diabetes & age_at_dx.notna(),
    (current_age - age_at_dx).clip(lower=0),  # floor at 0 to prevent negatives
    np.nan
)

# ── Smoking Category ──────────────────────────────────────────────────────────
df["Smoking_category"] = np.select(
    [
        df["SMQ020"] == 2,
        (df["SMQ020"] == 1) & (df["SMQ040"] == 3)
    ],
    ["Ideal", "Intermediate"],
    default="Poor"
)

# ── Alcohol Status ────────────────────────────────────────────────────────────
df["Alcohol_status"] = df["ALQ101"].map({
    1: "Alcohol_consumer",
    2: "Non_consumer"
})
df["Alcohol_status"] = pd.Categorical(
    df["Alcohol_status"],
    categories=["Non_consumer", "Alcohol_consumer"],
    ordered=True
)

# ── Physical Activity Category ────────────────────────────────────────────────


# Days per week: valid range 0–7; replace refusal/unknown codes with NaN
day_cols = ["PAQ610", "PAQ655", "PAQ625", "PAQ670"]
for col in day_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].where(df[col] <= 7)       # 7 = Refused, 9 = Don't know → NaN

# Minutes per day: replace sentinel codes with NaN
min_cols = ["PAD615", "PAD660", "PAD630", "PAD675"]
for col in min_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].where(df[col] < 7777)     # 7777 = Refused, 9999 = Don't know

# Vigorous minutes/week (work + recreational)
df["vigorous_min_week"] = (
    df["PAQ610"].fillna(0) * df["PAD615"].fillna(0) +
    df["PAQ655"].fillna(0) * df["PAD660"].fillna(0)
)

# Moderate minutes/week (walking/biking + recreational moderate)
df["moderate_min_week"] = (
    df["PAQ625"].fillna(0) * df["PAD630"].fillna(0) +
    df["PAQ670"].fillna(0) * df["PAD675"].fillna(0)
)

# MVPA equivalent (vigorous counts double per WHO/AHA guidelines)
df["mvpa_equivalent"] = df["moderate_min_week"] + 2 * df["vigorous_min_week"]

# Mark as NaN if all activity columns were missing (no data vs. zero activity)
all_missing = df[day_cols + min_cols].isna().all(axis=1)
df["mvpa_equivalent"] = df["mvpa_equivalent"].where(~all_missing, other=np.nan)

df["Physical_activity"] = np.select(
    [
        df["mvpa_equivalent"] >= 150,
        df["mvpa_equivalent"] > 0
    ],
    ["Ideal", "Intermediate"],
    default="Poor"
)

# ── Select & Rename Final Columns ─────────────────────────────────────────────

final = df[[
    "SEQN", "RIDAGEYR", "RIAGENDR", "RIDEXPRG", "RIDRETH1", "DMDEDUC2", "INDFMPIR",
    "BMXBMI", "SBP", "DBP", "LBXGH", "LBDLDL", "LBDHDD", "LBXTR",
    "LBDGLUSI", "LBXIN", "HOMA_IR", "HOMA_B", "LBDSGLSI", "LBDGLTSI",
    "LBXSCR", "eGFR", "PHAFSTHR",
    "DIQ010", "DID040","DIQ050", "DIQ070", "DIAB_DUR",          # ← diabetes block
    "Smoking_category", "Alcohol_status", "Physical_activity"
]].rename(columns={
    "RIDAGEYR":  "Age",
    "RIAGENDR":  "Sex",
    "RIDEXPRG":  "Pregnancy",
    "RIDRETH1":  "Ethnicity",
    "DMDEDUC2":  "Education_level",
    "INDFMPIR":  "Family_PIR",
    "BMXBMI":    "BMI",
    "SBP":       "Systolic_BP",
    "DBP":       "Diastolic_BP",
    "LBXGH":     "HbA1c",
    "LBDLDL":    "LDL",
    "LBDHDD":    "HDL",
    "LBXTR":     "Triglycerides",
    "LBDGLUSI":  "Fasting_glucose",
    "LBXIN":     "Fasting_insulin",
    "LBDSGLSI":   "Glucose",
    "LBDGLTSI":   "Two_hour_glucose",
    "LBXSCR":    "Creatinine",
    "PHAFSTHR":  "Fasting_hours",
    "DIQ010":    "Diabetes",
    "DIQ050":    "Insulin_pill",
    "DIQ070":     "Diabetes_pill",
    "DID040":    "Age_at_diagnosis",          # ← added
    "DIAB_DUR":  "Diabetes_duration"          # ← new
})

# ── Recode Ethnicity ──────────────────────────────────────────────────────────
ethnicity_map = {
    1: "Mexican American",
    2: "Other Hispanic",
    3: "Non-Hispanic White",
    4: "Non-Hispanic Black",
    5: "Other Race"
}
final["Ethnicity"] = final["Ethnicity"].map(ethnicity_map)

# ── Export ────────────────────────────────────────────────────────────────────
final.to_csv("nhanes_data_csvm/nhanes_2011_2012_clean.csv", index=False)
print("✅ Saved:", final.shape, "rows")
print(final[["Diabetes", "Age_at_diagnosis", "Diabetes_duration"]].dropna(subset=["Diabetes_duration"]).head(10))

## Cycle 8: NHANES 2013-2014

- Component files: suffix **`_H`**.
- **File reorganisation:** insulin (`LBXIN`) and fasting hours (`PHAFSTHR`) move into **`INS_H.XPT`**, out of the glucose file; the variable names themselves are unchanged, so only the read path differs.
- eGFR: CKD-EPI 2021. Physical activity: MVPA method. OGTT included. Age at diagnosis `DID040`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set the directory containing all .XPT files
BASE_DIR = Path(r"C:\Users\...\...\2013-2014")

# Load required files
demo     = pd.read_sas(BASE_DIR / "DEMO_H.XPT")
bmx      = pd.read_sas(BASE_DIR / "BMX_H.XPT")
bpx      = pd.read_sas(BASE_DIR / "BPX_H.XPT")
hdl      = pd.read_sas(BASE_DIR / "HDL_H.XPT")
trig     = pd.read_sas(BASE_DIR / "TRIGLY_H.XPT")
hba1c    = pd.read_sas(BASE_DIR / "GHB_H.XPT")
glu      = pd.read_sas(BASE_DIR / "GLU_H.XPT")
creat    = pd.read_sas(BASE_DIR / "BIOPRO_H.XPT")
# NOTE: From 2013-2014 onward, NHANES moved insulin (LBXIN) and fasting hours
#       (PHAFSTHR) out of the glucose file and into a dedicated INS_H.XPT file.
#       The variable names are unchanged — only the source file differs.
fast     = pd.read_sas(BASE_DIR / "INS_H.XPT")
ogtt     = pd.read_sas(BASE_DIR / "OGTT_H.XPT")
diabetes = pd.read_sas(BASE_DIR / "DIQ_H.XPT")
smoking  = pd.read_sas(BASE_DIR / "SMQ_H.XPT")
alcohol  = pd.read_sas(BASE_DIR / "ALQ_H.XPT")
activity = pd.read_sas(BASE_DIR / "PAQ_H.XPT")

# ── Select Relevant Features ──────────────────────────────────────────────────
demo     = demo[["SEQN", "RIDAGEYR", "RIAGENDR", "RIDEXPRG", "RIDRETH1", "DMDEDUC2", "INDFMPIR"]]
bmx      = bmx[["SEQN", "BMXBMI"]]
bpx      = bpx[["SEQN", "BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4",
                        "BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]]
hba1c    = hba1c[["SEQN", "LBXGH"]]
trig     = trig[["SEQN", "LBDLDL", "LBXTR"]]
hdl      = hdl[["SEQN", "LBDHDD"]]
glu      = glu[["SEQN", "LBDGLUSI"]]
creat    = creat[["SEQN", "LBXSCR", "LBDSGLSI"]]
# LBXIN (insulin) and PHAFSTHR (fasting hours) are both in INS_H.XPT this cycle
fast     = fast[["SEQN", "LBXIN", "PHAFSTHR"]]
ogtt     = ogtt[["SEQN", "LBDGLTSI"]]
# DID040 correctly selected in this cycle
diabetes = diabetes[["SEQN", "DIQ010", "DID040", "DIQ050", "DIQ070"]]
smoking  = smoking[["SEQN", "SMQ020", "SMQ040"]]
alcohol  = alcohol[["SEQN", "ALQ101"]]
activity = activity[["SEQN", "PAQ610", "PAD615", "PAQ655", "PAD660",
                             "PAQ625", "PAD630", "PAQ670", "PAD675"]]

# ── Mean Blood Pressure ───────────────────────────────────────────────────────
bpx["SBP"] = bpx[["BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4"]].mean(axis=1, skipna=True)
bpx["DBP"] = bpx[["BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]].mean(axis=1, skipna=True)

# ── eGFR CKD-EPI (2021) ───────────────────────────────────────────────────────
def compute_egfr_2021(creatinine_mgdl, age, gender):
    age             = pd.to_numeric(age,             errors='coerce')
    gender          = pd.to_numeric(gender,          errors='coerce')
    creatinine_mgdl = pd.to_numeric(creatinine_mgdl, errors='coerce')

    k         = np.where(gender == 1, 0.9,    0.7)
    alpha     = np.where(gender == 1, -0.302, -0.241)
    sex_coeff = np.where(gender == 1, 1.0,    1.012)

    min_scr = np.minimum(creatinine_mgdl / k, 1)
    max_scr = np.maximum(creatinine_mgdl / k, 1)

    egfr = 142 * (min_scr ** alpha) * (max_scr ** -1.200) * (0.9938 ** age) * sex_coeff
    return egfr

# ── HOMA-IR and HOMA-B ────────────────────────────────────────────────────────
def compute_homa(glu, ins, fasting_hours):
    glu  = pd.to_numeric(glu,           errors='coerce')
    ins  = pd.to_numeric(ins,           errors='coerce')
    # BUG FIX 1: Original compared the raw `fasting_hours` parameter before
    #            numeric conversion. Use the converted `fast` variable instead.
    fast = pd.to_numeric(fasting_hours, errors='coerce')

    if glu.median(skipna=True) > 30:   # Likely mg/dL → convert to mmol/L
        glu = glu / 18.018

    mask_fasting = fast >= 8           # ← fixed: was `fasting_hours >= 8`
    homa_ir = np.where(mask_fasting, (glu * ins) / 22.5, np.nan)
    homa_b  = np.where(mask_fasting & (glu > 3.5), (20 * ins) / (glu - 3.5), np.nan)
    return homa_ir, homa_b

# ── Merge all DataFrames on SEQN ──────────────────────────────────────────────
df = (
    demo
    .merge(bmx,                         on="SEQN", how="left")
    .merge(bpx[["SEQN", "SBP", "DBP"]], on="SEQN", how="left")
    .merge(hba1c,                       on="SEQN", how="left")
    .merge(trig,                        on="SEQN", how="left")
    .merge(glu,                         on="SEQN", how="left")
    .merge(hdl,                         on="SEQN", how="left")
    .merge(creat,                       on="SEQN", how="left")
    .merge(fast,                        on="SEQN", how="left")
    .merge(ogtt,                        on="SEQN", how="left")
    .merge(diabetes,                    on="SEQN", how="left")
    .merge(smoking,                     on="SEQN", how="left")
    .merge(alcohol,                     on="SEQN", how="left")
    .merge(activity,                    on="SEQN", how="left")
)

# ── Derived Variables ─────────────────────────────────────────────────────────
df["eGFR"]                  = compute_egfr_2021(df["LBXSCR"], df["RIDAGEYR"], df["RIAGENDR"])
df["HOMA_IR"], df["HOMA_B"] = compute_homa(df["LBDGLUSI"], df["LBXIN"], df["PHAFSTHR"])

# ── Diabetes Duration Workflow ────────────────────────────────────────────────
# Step 1 – Identify participants with confirmed diabetes (DIQ010 = 1)
has_diabetes = df["DIQ010"] == 1

# Step 2 – Extract DID040 (age at diagnosis); sentinel values
#           (777 = Refused, 999 = Don't know) and implausible ages → NaN
age_at_dx = pd.to_numeric(df["DID040"], errors='coerce')
age_at_dx = age_at_dx.where(age_at_dx < 120)

# Step 3 – Current age as numeric
current_age = pd.to_numeric(df["RIDAGEYR"], errors='coerce')

# Step 4 – DIAB_DUR: only populated for confirmed diabetics with a valid diagnosis age
df["DIAB_DUR"] = np.where(
    has_diabetes & age_at_dx.notna(),
    (current_age - age_at_dx).clip(lower=0),  # floor at 0 to prevent negatives
    np.nan
)

# ── Smoking Category ──────────────────────────────────────────────────────────
df["Smoking_category"] = np.select(
    [
        df["SMQ020"] == 2,
        (df["SMQ020"] == 1) & (df["SMQ040"] == 3)
    ],
    ["Ideal", "Intermediate"],
    default="Poor"
)

# ── Alcohol Status ────────────────────────────────────────────────────────────
df["Alcohol_status"] = df["ALQ101"].map({
    1: "Alcohol_consumer",
    2: "Non_consumer"
})
df["Alcohol_status"] = pd.Categorical(
    df["Alcohol_status"],
    categories=["Non_consumer", "Alcohol_consumer"],
    ordered=True
)

# ── Physical Activity Category ────────────────────────────────────────────────


# Days per week: valid range 0–7; replace refusal/unknown codes with NaN
day_cols = ["PAQ610", "PAQ655", "PAQ625", "PAQ670"]
for col in day_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].where(df[col] <= 7)       # 7 = Refused, 9 = Don't know → NaN

# Minutes per day: replace sentinel codes with NaN
min_cols = ["PAD615", "PAD660", "PAD630", "PAD675"]
for col in min_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].where(df[col] < 7777)     # 7777 = Refused, 9999 = Don't know

# Vigorous minutes/week (work + recreational)
df["vigorous_min_week"] = (
    df["PAQ610"].fillna(0) * df["PAD615"].fillna(0) +
    df["PAQ655"].fillna(0) * df["PAD660"].fillna(0)
)

# Moderate minutes/week (walking/biking + recreational moderate)
df["moderate_min_week"] = (
    df["PAQ625"].fillna(0) * df["PAD630"].fillna(0) +
    df["PAQ670"].fillna(0) * df["PAD675"].fillna(0)
)

# MVPA equivalent (vigorous counts double per WHO/AHA guidelines)
df["mvpa_equivalent"] = df["moderate_min_week"] + 2 * df["vigorous_min_week"]

# Mark as NaN if all activity columns were missing (no data vs. zero activity)
all_missing = df[day_cols + min_cols].isna().all(axis=1)
df["mvpa_equivalent"] = df["mvpa_equivalent"].where(~all_missing, other=np.nan)

df["Physical_activity"] = np.select(
    [
        df["mvpa_equivalent"] >= 150,
        df["mvpa_equivalent"] > 0
    ],
    ["Ideal", "Intermediate"],
    default="Poor"
)

# ── Select & Rename Final Columns ─────────────────────────────────────────────

final = df[[
    "SEQN", "RIDAGEYR", "RIAGENDR", "RIDEXPRG", "RIDRETH1", "DMDEDUC2", "INDFMPIR",
    "BMXBMI", "SBP", "DBP", "LBXGH", "LBDLDL", "LBDHDD", "LBXTR",
    "LBDGLUSI", "LBXIN", "HOMA_IR", "HOMA_B", "LBDSGLSI", "LBDGLTSI",
    "LBXSCR", "eGFR", "PHAFSTHR",
    "DIQ010", "DID040", "DIQ050", "DIQ070", "DIAB_DUR",          # ← diabetes block
    "Smoking_category", "Alcohol_status", "Physical_activity"
]].rename(columns={
    "RIDAGEYR":  "Age",
    "RIAGENDR":  "Sex",
    "RIDEXPRG":  "Pregnancy",
    "RIDRETH1":  "Ethnicity",
    "DMDEDUC2":  "Education_level",
    "INDFMPIR":  "Family_PIR",
    "BMXBMI":    "BMI",
    "SBP":       "Systolic_BP",
    "DBP":       "Diastolic_BP",
    "LBXGH":     "HbA1c",
    "LBDLDL":    "LDL",
    "LBDHDD":    "HDL",
    "LBXTR":     "Triglycerides",
    "LBDGLUSI":  "Fasting_glucose",
    "LBXIN":     "Fasting_insulin",
    "LBDSGLSI":  "Glucose",
    "LBDGLTSI":  "Two_hour_glucose",
    "LBXSCR":    "Creatinine",
    "PHAFSTHR":  "Fasting_hours",
    "DIQ010":    "Diabetes",
    "DIQ050":     "Insulin_pill",
    "DIQ070":     "Diabetes_pill",
    "DID040":    "Age_at_diagnosis",          # ← added
    "DIAB_DUR":  "Diabetes_duration"          # ← new
})

# ── Recode Ethnicity ──────────────────────────────────────────────────────────
ethnicity_map = {
    1: "Mexican American",
    2: "Other Hispanic",
    3: "Non-Hispanic White",
    4: "Non-Hispanic Black",
    5: "Other Race"
}
final["Ethnicity"] = final["Ethnicity"].map(ethnicity_map)

# ── Export ────────────────────────────────────────────────────────────────────
final.to_csv("nhanes_data_csvm/nhanes_2013_2014_clean.csv", index=False)
print("✅ Saved:", final.shape, "rows")
print(final[["Diabetes", "Age_at_diagnosis", "Diabetes_duration"]].dropna(subset=["Diabetes_duration"]).head(10))

## Cycle 9: NHANES 2015-2016

- Component files: suffix **`_I`**.
- Insulin and fasting hours in **`INS_I.XPT`**, as in 2013-2014. Otherwise identical in structure.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set the directory containing all .XPT files
BASE_DIR = Path(r"C:\Users\...\...\2015-2016")

# Load required files
demo     = pd.read_sas(BASE_DIR / "DEMO_I.XPT")
bmx      = pd.read_sas(BASE_DIR / "BMX_I.XPT")
bpx      = pd.read_sas(BASE_DIR / "BPX_I.XPT")
hdl      = pd.read_sas(BASE_DIR / "HDL_I.XPT")
trig     = pd.read_sas(BASE_DIR / "TRIGLY_I.XPT")
hba1c    = pd.read_sas(BASE_DIR / "GHB_I.XPT")
glu      = pd.read_sas(BASE_DIR / "GLU_I.XPT")
creat    = pd.read_sas(BASE_DIR / "BIOPRO_I.XPT")
# LBXIN (insulin) and PHAFSTHR (fasting hours) are in INS_I.XPT from 2013 onward
fast     = pd.read_sas(BASE_DIR / "INS_I.XPT")
ogtt     = pd.read_sas(BASE_DIR / "OGTT_I.XPT")
diabetes = pd.read_sas(BASE_DIR / "DIQ_I.XPT")
smoking  = pd.read_sas(BASE_DIR / "SMQ_I.XPT")
alcohol  = pd.read_sas(BASE_DIR / "ALQ_I.XPT")
activity = pd.read_sas(BASE_DIR / "PAQ_I.XPT")

# ── Select Relevant Features ──────────────────────────────────────────────────
demo     = demo[["SEQN", "RIDAGEYR", "RIAGENDR", "RIDEXPRG", "RIDRETH1", "DMDEDUC2", "INDFMPIR"]]
bmx      = bmx[["SEQN", "BMXBMI"]]
bpx      = bpx[["SEQN", "BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4",
                        "BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]]
hba1c    = hba1c[["SEQN", "LBXGH"]]
trig     = trig[["SEQN", "LBDLDL", "LBXTR"]]
hdl      = hdl[["SEQN", "LBDHDD"]]
glu      = glu[["SEQN", "LBDGLUSI"]]
creat    = creat[["SEQN", "LBXSCR", "LBDSGLSI"]]
fast     = fast[["SEQN", "LBXIN", "PHAFSTHR"]]
ogtt     = ogtt[["SEQN", "LBDGLTSI"]]

# DID040 correctly selected in this cycle
diabetes = diabetes[["SEQN", "DIQ010", "DID040", "DIQ050", "DIQ070"]]
smoking  = smoking[["SEQN", "SMQ020", "SMQ040"]]
alcohol  = alcohol[["SEQN", "ALQ101"]]
activity = activity[["SEQN", "PAQ610", "PAD615", "PAQ655", "PAD660",
                             "PAQ625", "PAD630", "PAQ670", "PAD675"]]

# ── Mean Blood Pressure ───────────────────────────────────────────────────────
bpx["SBP"] = bpx[["BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4"]].mean(axis=1, skipna=True)
bpx["DBP"] = bpx[["BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]].mean(axis=1, skipna=True)

# ── eGFR CKD-EPI (2021) ───────────────────────────────────────────────────────
def compute_egfr_2021(creatinine_mgdl, age, gender):
    age             = pd.to_numeric(age,             errors='coerce')
    gender          = pd.to_numeric(gender,          errors='coerce')
    creatinine_mgdl = pd.to_numeric(creatinine_mgdl, errors='coerce')

    k         = np.where(gender == 1, 0.9,    0.7)
    alpha     = np.where(gender == 1, -0.302, -0.241)
    sex_coeff = np.where(gender == 1, 1.0,    1.012)

    min_scr = np.minimum(creatinine_mgdl / k, 1)
    max_scr = np.maximum(creatinine_mgdl / k, 1)

    egfr = 142 * (min_scr ** alpha) * (max_scr ** -1.200) * (0.9938 ** age) * sex_coeff
    return egfr

# ── HOMA-IR and HOMA-B ────────────────────────────────────────────────────────
def compute_homa(glu, ins, fasting_hours):
    glu  = pd.to_numeric(glu,           errors='coerce')
    ins  = pd.to_numeric(ins,           errors='coerce')
    # BUG FIX 1: Original compared the raw `fasting_hours` parameter before
    #            numeric conversion. Use the converted `fast` variable instead.
    fast = pd.to_numeric(fasting_hours, errors='coerce')

    if glu.median(skipna=True) > 30:   # Likely mg/dL → convert to mmol/L
        glu = glu / 18.018

    mask_fasting = fast >= 8           # ← fixed: was `fasting_hours >= 8`
    homa_ir = np.where(mask_fasting, (glu * ins) / 22.5, np.nan)
    homa_b  = np.where(mask_fasting & (glu > 3.5), (20 * ins) / (glu - 3.5), np.nan)
    return homa_ir, homa_b

# ── Merge all DataFrames on SEQN ──────────────────────────────────────────────
df = (
    demo
    .merge(bmx,                         on="SEQN", how="left")
    .merge(bpx[["SEQN", "SBP", "DBP"]], on="SEQN", how="left")
    .merge(hba1c,                       on="SEQN", how="left")
    .merge(trig,                        on="SEQN", how="left")
    .merge(glu,                         on="SEQN", how="left")
    .merge(hdl,                         on="SEQN", how="left")
    .merge(creat,                       on="SEQN", how="left")
    .merge(fast,                        on="SEQN", how="left")
    .merge(ogtt,                        on="SEQN", how="left")
    .merge(diabetes,                    on="SEQN", how="left")
    .merge(smoking,                     on="SEQN", how="left")
    .merge(alcohol,                     on="SEQN", how="left")
    .merge(activity,                    on="SEQN", how="left")
)

# ── Derived Variables ─────────────────────────────────────────────────────────
df["eGFR"]                  = compute_egfr_2021(df["LBXSCR"], df["RIDAGEYR"], df["RIAGENDR"])
df["HOMA_IR"], df["HOMA_B"] = compute_homa(df["LBDGLUSI"], df["LBXIN"], df["PHAFSTHR"])

# ── Diabetes Duration Workflow ────────────────────────────────────────────────
# Step 1 – Identify participants with confirmed diabetes (DIQ010 = 1)
has_diabetes = df["DIQ010"] == 1

# Step 2 – Extract DID040 (age at diagnosis); sentinel values
#           (777 = Refused, 999 = Don't know) and implausible ages → NaN
age_at_dx = pd.to_numeric(df["DID040"], errors='coerce')
age_at_dx = age_at_dx.where(age_at_dx < 120)

# Step 3 – Current age as numeric
current_age = pd.to_numeric(df["RIDAGEYR"], errors='coerce')

# Step 4 – DIAB_DUR: only populated for confirmed diabetics with a valid diagnosis age
df["DIAB_DUR"] = np.where(
    has_diabetes & age_at_dx.notna(),
    (current_age - age_at_dx).clip(lower=0),  # floor at 0 to prevent negatives
    np.nan
)

# ── Smoking Category ──────────────────────────────────────────────────────────
df["Smoking_category"] = np.select(
    [
        df["SMQ020"] == 2,
        (df["SMQ020"] == 1) & (df["SMQ040"] == 3)
    ],
    ["Ideal", "Intermediate"],
    default="Poor"
)

# ── Alcohol Status ────────────────────────────────────────────────────────────
df["Alcohol_status"] = df["ALQ101"].map({
    1: "Alcohol_consumer",
    2: "Non_consumer"
})
df["Alcohol_status"] = pd.Categorical(
    df["Alcohol_status"],
    categories=["Non_consumer", "Alcohol_consumer"],
    ordered=True
)

# ── Physical Activity Category ────────────────────────────────────────────────


# Days per week: valid range 0–7; replace refusal/unknown codes with NaN
day_cols = ["PAQ610", "PAQ655", "PAQ625", "PAQ670"]
for col in day_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].where(df[col] <= 7)       # 7 = Refused, 9 = Don't know → NaN

# Minutes per day: replace sentinel codes with NaN
min_cols = ["PAD615", "PAD660", "PAD630", "PAD675"]
for col in min_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].where(df[col] < 7777)     # 7777 = Refused, 9999 = Don't know

# Vigorous minutes/week (work + recreational)
df["vigorous_min_week"] = (
    df["PAQ610"].fillna(0) * df["PAD615"].fillna(0) +
    df["PAQ655"].fillna(0) * df["PAD660"].fillna(0)
)

# Moderate minutes/week (walking/biking + recreational moderate)
df["moderate_min_week"] = (
    df["PAQ625"].fillna(0) * df["PAD630"].fillna(0) +
    df["PAQ670"].fillna(0) * df["PAD675"].fillna(0)
)

# MVPA equivalent (vigorous counts double per WHO/AHA guidelines)
df["mvpa_equivalent"] = df["moderate_min_week"] + 2 * df["vigorous_min_week"]

# Mark as NaN if all activity columns were missing (no data vs. zero activity)
all_missing = df[day_cols + min_cols].isna().all(axis=1)
df["mvpa_equivalent"] = df["mvpa_equivalent"].where(~all_missing, other=np.nan)

df["Physical_activity"] = np.select(
    [
        df["mvpa_equivalent"] >= 150,
        df["mvpa_equivalent"] > 0
    ],
    ["Ideal", "Intermediate"],
    default="Poor"
)

# ── Select & Rename Final Columns ─────────────────────────────────────────────

final = df[[
    "SEQN", "RIDAGEYR", "RIAGENDR", "RIDEXPRG", "RIDRETH1", "DMDEDUC2", "INDFMPIR",
    "BMXBMI", "SBP", "DBP", "LBXGH", "LBDLDL", "LBDHDD", "LBXTR",
    "LBDGLUSI", "LBXIN", "HOMA_IR", "HOMA_B", "LBDSGLSI", "LBDGLTSI",
    "LBXSCR", "eGFR", "PHAFSTHR",
    "DIQ010", "DID040", "DIQ050", "DIQ070", "DIAB_DUR",          # ← diabetes block
    "Smoking_category", "Alcohol_status", "Physical_activity"
]].rename(columns={
    "RIDAGEYR":  "Age",
    "RIAGENDR":  "Sex",
    "RIDEXPRG":  "Pregnancy",
    "RIDRETH1":  "Ethnicity",
    "DMDEDUC2":  "Education_level",
    "INDFMPIR":  "Family_PIR",
    "BMXBMI":    "BMI",
    "SBP":       "Systolic_BP",
    "DBP":       "Diastolic_BP",
    "LBXGH":     "HbA1c",
    "LBDLDL":    "LDL",
    "LBDHDD":    "HDL",
    "LBXTR":     "Triglycerides",
    "LBDGLUSI":  "Fasting_glucose",
    "LBXIN":     "Fasting_insulin",
    "LBDSGLSI":   "Glucose",
    "LBDGLTSI":   "Two_hour_glucose",
    "LBXSCR":    "Creatinine",
    "PHAFSTHR":  "Fasting_hours",
    "DIQ010":    "Diabetes",
    "DIQ050":    "Insulin_pill",
    "DIQ070":    "Diabetes_pill",
    "DID040":    "Age_at_diagnosis",          # ← added
    "DIAB_DUR":  "Diabetes_duration"          # ← new
})

# ── Recode Ethnicity ──────────────────────────────────────────────────────────
ethnicity_map = {
    1: "Mexican American",
    2: "Other Hispanic",
    3: "Non-Hispanic White",
    4: "Non-Hispanic Black",
    5: "Other Race"
}
final["Ethnicity"] = final["Ethnicity"].map(ethnicity_map)

# ── Export ────────────────────────────────────────────────────────────────────
final.to_csv("nhanes_data_csvm/nhanes_2015_2016_clean.csv", index=False)
print("✅ Saved:", final.shape, "rows")
print(final[["Diabetes", "Age_at_diagnosis", "Diabetes_duration"]].dropna(subset=["Diabetes_duration"]).head(10))

## Cycle 10: NHANES 2017-2018

- Component files: suffix **`_J`**.
- **File reorganisation again:** fasting hours in **`FASTQX_J.XPT`** and insulin in **`INS_J.XPT`** - two separate merges this cycle.
- **Alcohol is derived from `ALQ121`** because `ALQ101` was dropped: drinking at least monthly (codes 1-7) maps to consumer, less often or never to non-consumer. `np.select` is used rather than `np.where` to mix `None` with strings without a dtype clash.
- OGTT is **not** merged here, so `Two_hour_glucose` is absent from this cycle's final table. Age at diagnosis `DID040`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set the directory containing all .XPT files
BASE_DIR = Path(r"C:\Users\...\...\2017-2018")

# Load required files
demo     = pd.read_sas(BASE_DIR / "DEMO_J.XPT")
bmx      = pd.read_sas(BASE_DIR / "BMX_J.XPT")
bpx      = pd.read_sas(BASE_DIR / "BPX_J.XPT")
hdl      = pd.read_sas(BASE_DIR / "HDL_J.XPT")
trig     = pd.read_sas(BASE_DIR / "TRIGLY_J.XPT")
hba1c    = pd.read_sas(BASE_DIR / "GHB_J.XPT")
glu      = pd.read_sas(BASE_DIR / "GLU_J.XPT")
creat    = pd.read_sas(BASE_DIR / "BIOPRO_J.XPT")
# NOTE: From 2017-2018 onward, fasting hours (PHAFSTHR) moved back to FASTQX_J.XPT
#       while insulin (LBXIN) remains in INS_J.XPT — they are now separate files again.
fast     = pd.read_sas(BASE_DIR / "FASTQX_J.XPT")

ins      = pd.read_sas(BASE_DIR / "INS_J.XPT")
diabetes = pd.read_sas(BASE_DIR / "DIQ_J.XPT")
smoking  = pd.read_sas(BASE_DIR / "SMQ_J.XPT")
alcohol  = pd.read_sas(BASE_DIR / "ALQ_J.XPT")
activity = pd.read_sas(BASE_DIR / "PAQ_J.XPT")

# ── Select Relevant Features ──────────────────────────────────────────────────
demo     = demo[["SEQN", "RIDAGEYR", "RIAGENDR", "RIDEXPRG", "RIDRETH1", "DMDEDUC2", "INDFMPIR"]]
bmx      = bmx[["SEQN", "BMXBMI"]]
bpx      = bpx[["SEQN", "BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4",
                        "BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]]
hba1c    = hba1c[["SEQN", "LBXGH"]]
trig     = trig[["SEQN", "LBDLDL", "LBXTR"]]
hdl      = hdl[["SEQN", "LBDHDD"]]
glu      = glu[["SEQN", "LBDGLUSI"]]
creat    = creat[["SEQN", "LBXSCR", "LBDSGLSI"]]
fast     = fast[["SEQN", "PHAFSTHR"]]
ins      = ins[["SEQN", "LBXIN"]]

diabetes = diabetes[["SEQN", "DIQ010", "DID040", "DIQ050", "DIQ070"]]
smoking  = smoking[["SEQN", "SMQ020", "SMQ040"]]

# ── Alcohol Status (2017-2018) ────────────────────────────────────────────────
# ALQ101 (used in 1999-2015) asked "≥12 drinks in the past year?" (1=Yes/2=No).
# This question was dropped in 2017-2018. The closest comparable derivation is
# from ALQ121 ("how often did you drink in the past 12 months?"):
#   - Once/month or more (codes 1–7) → ≥12 drinks/year → Alcohol_consumer
#   - Less than monthly  (codes 8–10) → <12 drinks/year → Non_consumer
#   - Never              (code 0)     →                 → Non_consumer
#   - Sentinel codes 777 (Refused) / 999 (Don't know)  → NaN

alcohol  = alcohol[["SEQN", "ALQ121"]]

activity = activity[["SEQN", "PAQ610", "PAD615", "PAQ655", "PAD660",
                             "PAQ625", "PAD630", "PAQ670", "PAD675"]]

# ── Mean Blood Pressure ───────────────────────────────────────────────────────
bpx["SBP"] = bpx[["BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4"]].mean(axis=1, skipna=True)
bpx["DBP"] = bpx[["BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]].mean(axis=1, skipna=True)

# ── eGFR CKD-EPI (2021) ───────────────────────────────────────────────────────
def compute_egfr_2021(creatinine_mgdl, age, gender):
    age             = pd.to_numeric(age,             errors='coerce')
    gender          = pd.to_numeric(gender,          errors='coerce')
    creatinine_mgdl = pd.to_numeric(creatinine_mgdl, errors='coerce')

    k         = np.where(gender == 1, 0.9,    0.7)
    alpha     = np.where(gender == 1, -0.302, -0.241)
    sex_coeff = np.where(gender == 1, 1.0,    1.012)

    min_scr = np.minimum(creatinine_mgdl / k, 1)
    max_scr = np.maximum(creatinine_mgdl / k, 1)

    egfr = 142 * (min_scr ** alpha) * (max_scr ** -1.200) * (0.9938 ** age) * sex_coeff
    return egfr

# ── HOMA-IR and HOMA-B ────────────────────────────────────────────────────────
def compute_homa(glu, ins, fasting_hours):
    glu  = pd.to_numeric(glu,           errors='coerce')
    ins  = pd.to_numeric(ins,           errors='coerce')

    fast = pd.to_numeric(fasting_hours, errors='coerce')

    if glu.median(skipna=True) > 30:   # Likely mg/dL → convert to mmol/L
        glu = glu / 18.018

    mask_fasting = fast >= 8           # ← fixed: was `fasting_hours >= 8`
    homa_ir = np.where(mask_fasting, (glu * ins) / 22.5, np.nan)
    homa_b  = np.where(mask_fasting & (glu > 3.5), (20 * ins) / (glu - 3.5), np.nan)
    return homa_ir, homa_b

# ── Merge all DataFrames on SEQN ──────────────────────────────────────────────
df = (
    demo
    .merge(bmx,                         on="SEQN", how="left")
    .merge(bpx[["SEQN", "SBP", "DBP"]], on="SEQN", how="left")
    .merge(hba1c,                       on="SEQN", how="left")
    .merge(trig,                        on="SEQN", how="left")
    .merge(glu,                         on="SEQN", how="left")
    .merge(hdl,                         on="SEQN", how="left")
    .merge(creat,                       on="SEQN", how="left")
    .merge(ins,                         on="SEQN", how="left")
    .merge(fast,                        on="SEQN", how="left")
    .merge(diabetes,                    on="SEQN", how="left")
    .merge(smoking,                     on="SEQN", how="left")
    .merge(alcohol,                     on="SEQN", how="left")
    .merge(activity,                    on="SEQN", how="left")
)

# ── Derived Variables ─────────────────────────────────────────────────────────
df["eGFR"]                  = compute_egfr_2021(df["LBXSCR"], df["RIDAGEYR"], df["RIAGENDR"])
df["HOMA_IR"], df["HOMA_B"] = compute_homa(df["LBDGLUSI"], df["LBXIN"], df["PHAFSTHR"])

# ── Diabetes Duration Workflow ────────────────────────────────────────────────
# Step 1 – Identify participants with confirmed diabetes (DIQ010 = 1)
has_diabetes = df["DIQ010"] == 1

# Step 2 – Extract DID040 (age at diagnosis); sentinel values
#           (777 = Refused, 999 = Don't know) and implausible ages → NaN
age_at_dx = pd.to_numeric(df["DID040"], errors='coerce')
age_at_dx = age_at_dx.where(age_at_dx < 120)

# Step 3 – Current age as numeric
current_age = pd.to_numeric(df["RIDAGEYR"], errors='coerce')

# Step 4 – DIAB_DUR: only populated for confirmed diabetics with a valid diagnosis age
df["DIAB_DUR"] = np.where(
    has_diabetes & age_at_dx.notna(),
    (current_age - age_at_dx).clip(lower=0),  # floor at 0 to prevent negatives
    np.nan
)

# ── Smoking Category ──────────────────────────────────────────────────────────
df["Smoking_category"] = np.select(
    [
        df["SMQ020"] == 2,
        (df["SMQ020"] == 1) & (df["SMQ040"] == 3)
    ],
    ["Ideal", "Intermediate"],
    default="Poor"
)

# ── Alcohol Status ────────────────────────────────────────────────────────────

# np.select handles mixed string/NaN outputs cleanly — avoids the
# DTypePromotionError caused by mixing np.nan (float) with strings in np.where

df["Alcohol_status"] = np.select(
    [
        df["ALQ121"].isna(),              # Refused / Don't know / missing
        df["ALQ121"].between(1, 7),       # ≥ once/month → ≥12 drinks/year
    ],
    [
        None,                             # → NaN (no type conflict with np.select)
        "Alcohol_consumer",
    ],
    default="Non_consumer"                # codes 0, 8, 9, 10 → <12 drinks/year
)

df["Alcohol_status"] = pd.Categorical(
    df["Alcohol_status"],
    categories=["Non_consumer", "Alcohol_consumer"],
    ordered=True
)

# ── Physical Activity Category ────────────────────────────────────────────────
# BUG FIX 3: NHANES PAQ* day-count variables use 7 (Refused) / 9 (Don't know)
#            and PAD* minute variables use 7777 / 9999 as sentinel codes.
#            Without replacing these with NaN first, refused/unknown responses
#            silently inflate MVPA totals and misclassify participants.

# Days per week: valid range 0–7; replace refusal/unknown codes with NaN
day_cols = ["PAQ610", "PAQ655", "PAQ625", "PAQ670"]
for col in day_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].where(df[col] <= 7)       # 7 = Refused, 9 = Don't know → NaN

# Minutes per day: replace sentinel codes with NaN
min_cols = ["PAD615", "PAD660", "PAD630", "PAD675"]
for col in min_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].where(df[col] < 7777)     # 7777 = Refused, 9999 = Don't know

# Vigorous minutes/week (work + recreational)
df["vigorous_min_week"] = (
    df["PAQ610"].fillna(0) * df["PAD615"].fillna(0) +
    df["PAQ655"].fillna(0) * df["PAD660"].fillna(0)
)

# Moderate minutes/week (walking/biking + recreational moderate)
df["moderate_min_week"] = (
    df["PAQ625"].fillna(0) * df["PAD630"].fillna(0) +
    df["PAQ670"].fillna(0) * df["PAD675"].fillna(0)
)

# MVPA equivalent (vigorous counts double per WHO/AHA guidelines)
df["mvpa_equivalent"] = df["moderate_min_week"] + 2 * df["vigorous_min_week"]

# Mark as NaN if all activity columns were missing (no data vs. zero activity)
all_missing = df[day_cols + min_cols].isna().all(axis=1)
df["mvpa_equivalent"] = df["mvpa_equivalent"].where(~all_missing, other=np.nan)

df["Physical_activity"] = np.select(
    [
        df["mvpa_equivalent"] >= 150,
        df["mvpa_equivalent"] > 0
    ],
    ["Ideal", "Intermediate"],
    default="Poor"
)

# ── Select & Rename Final Columns ─────────────────────────────────────────────
# BUG FIX 4 & 5: DID040 (age at diagnosis) and DIAB_DUR were missing from the
#                final column list despite DID040 being correctly loaded above.
final = df[[
    "SEQN", "RIDAGEYR", "RIAGENDR", "RIDEXPRG", "RIDRETH1", "DMDEDUC2", "INDFMPIR",
    "BMXBMI", "SBP", "DBP", "LBXGH", "LBDLDL", "LBDHDD", "LBXTR",
    "LBDGLUSI", "LBXIN", "HOMA_IR", "HOMA_B", "LBDSGLSI",
    "LBXSCR", "eGFR", "PHAFSTHR",
    "DIQ010", "DID040","DIQ050", "DIQ070", "DIAB_DUR",          # ← diabetes block
    "Smoking_category", "Alcohol_status", "Physical_activity"
]].rename(columns={
    "RIDAGEYR":  "Age",
    "RIAGENDR":  "Sex",
    "RIDEXPRG":  "Pregnancy",
    "RIDRETH1":  "Ethnicity",
    "DMDEDUC2":  "Education_level",
    "INDFMPIR":  "Family_PIR",
    "BMXBMI":    "BMI",
    "SBP":       "Systolic_BP",
    "DBP":       "Diastolic_BP",
    "LBXGH":     "HbA1c",
    "LBDLDL":    "LDL",
    "LBDHDD":    "HDL",
    "LBXTR":     "Triglycerides",
    "LBDGLUSI":  "Fasting_glucose",
    "LBXIN":     "Fasting_insulin",
    "LBDSGLSI":   "Glucose",
    "LBXSCR":    "Creatinine",
    "PHAFSTHR":  "Fasting_hours",
    "DIQ010":    "Diabetes",
    "DIQ050":    "Insulin_pill",
    "DIQ070":    "Diabetes_pill",
    "DID040":    "Age_at_diagnosis",          # ← added
    "DIAB_DUR":  "Diabetes_duration"          # ← new
})

# ── Recode Ethnicity ──────────────────────────────────────────────────────────
ethnicity_map = {
    1: "Mexican American",
    2: "Other Hispanic",
    3: "Non-Hispanic White",
    4: "Non-Hispanic Black",
    5: "Other Race"
}
final["Ethnicity"] = final["Ethnicity"].map(ethnicity_map)

# ── Export ────────────────────────────────────────────────────────────────────
final.to_csv("nhanes_data_csvm/nhanes_2017_2018_clean.csv", index=False)
print("✅ Saved:", final.shape, "rows")
print(final[["Diabetes", "Age_at_diagnosis", "Diabetes_duration"]].dropna(subset=["Diabetes_duration"]).head(10))

## Combine cycles into a single dataset

- Loads the ten `nhanes_YYYY_YYYY_clean.csv` files, tags each with a `Cycle` label so cycle membership survives the stack, and concatenates them row-wise with `pd.concat(..., ignore_index=True)`.
- Result: **101,316 rows x 32 columns**. Columns are aligned by name; any column absent in a given cycle (for example `Two_hour_glucose` outside 2005-2016) is filled with `NaN` for that cycle's rows.
- The `Cycle` column is what makes it possible later to adjust for survey period or restrict to a consistent span for the variables noted as non-comparable above.
- REVIEW: read paths here are absolute (`C:/Users/...`) - adjust for your environment (see Known issues #3).

In [ ]:
import pandas as pd

# Load the two cleaned cycle files
df_1999_2000 = pd.read_csv(r"C:/Users/.../.../nhanes_data_csvm/nhanes_1999_2000_clean.csv")
df_2001_2002 = pd.read_csv(r"C:/Users/.../.../nhanes_data_csvm/nhanes_2001_2002_clean.csv")
df_2003_2004 = pd.read_csv(r"C:/Users/.../.../nhanes_data_csvm/nhanes_2003_2004_clean.csv")
df_2005_2006 = pd.read_csv(r"C:/Users/.../.../nhanes_data_csvm/nhanes_2005_2006_clean.csv")
df_2007_2008 = pd.read_csv(r"C:/Users/.../.../nhanes_data_csvm/nhanes_2007_2008_clean.csv")
df_2009_2010 = pd.read_csv(r"C:/Users/.../.../nhanes_data_csvm/nhanes_2009_2010_clean.csv")
df_2011_2012 = pd.read_csv(r"C:/Users/.../.../nhanes_data_csvm/nhanes_2011_2012_clean.csv")
df_2013_2014 = pd.read_csv(r"C:/Users/.../.../nhanes_data_csvm/nhanes_2013_2014_clean.csv")
df_2015_2016 = pd.read_csv(r"C:/Users/.../.../nhanes_data_csvm/nhanes_2015_2016_clean.csv")
df_2017_2018 = pd.read_csv(r"C:/Users/.../.../nhanes_data_csvm/nhanes_2017_2018_clean.csv")


# Add a column to indicate cycle (optional but recommended)
df_1999_2000["Cycle"] = "1999_2000"
df_2001_2002["Cycle"] = "2001_2002"
df_2003_2004["Cycle"] = "2003_2004"
df_2005_2006["Cycle"] = "2005_2006"
df_2007_2008["Cycle"] = "2007_2008"
df_2009_2010["Cycle"] = "2009_2010"
df_2011_2012["Cycle"] = "2011_2012"
df_2013_2014["Cycle"] = "2013_2014"
df_2015_2016["Cycle"] = "2015_2016"
df_2017_2018["Cycle"] = "2017_2018"

# Concatenate (stack) the two datasets vertically
combined = pd.concat([df_1999_2000, df_2001_2002, df_2003_2004, df_2005_2006, df_2007_2008, df_2009_2010, df_2011_2012, df_2013_2014, df_2015_2016, df_2017_2018],  ignore_index=True)


# Check result
print(combined.shape)
print(combined["Cycle"].value_counts())



# Save to new CSV
combined.to_csv("nhanes_1999_2018_combined_new.csv", index=False)
print("✅ Combined dataset saved!")


## Inspect the combined dataset

- Prints the schema, row count and non-null counts for the combined table.
- This is the main sanity check on the merge: confirm the row total matches the sum of the per-cycle counts, and that missingness follows the expected pattern - HOMA and OGTT variables populated only for the fasting / OGTT subsamples, `Age_at_diagnosis` and `Diabetes_duration` only for diagnosed participants.

In [ ]:
print(combined.info())